# AWS SageMaker, Part 2. Training and Serving from Code

Training a model with a SageMaker training job and serving it three ways, as a real-time endpoint, a serverless endpoint and a batch job. Everything is started from Python files in VS Code, on the private platform built in the previous class.

**How to use this notebook.** Copy directly from the visible code cells into the terminal, file, or AWS field named above each one. These cells are a copy-and-paste guide, not steps to execute in this notebook. Replace the indicated placeholders with your own values. Explanations, screenshots, and example outputs remain in rendered Markdown.

---

## **Agenda**

1. [Building the platform from nothing](#scrollTo=c003)
2. [A training job from code](#scrollTo=c008)
3. [A real-time endpoint](#scrollTo=c013)
4. [A serverless endpoint](#scrollTo=c028)
5. [Batch transform](#scrollTo=c031)
6. [Tearing it all down](#scrollTo=c034)
7. [Class summary](#scrollTo=c037)
8. [References and further reading](#scrollTo=c038)
9. [What's next](#scrollTo=c039)

## **Preface**

The previous class built the platform by hand. There was a private VPC with no way in from the internet, a role that can use one bucket, a VPC-only SageMaker domain and a space that VS Code connects to. It also got the data ready. `prepare.py` dropped the leaky `refund_amount` column and wrote four files to S3 in the format the built-in XGBoost algorithm wants.

This class does what SageMaker is mostly used for in real teams. A **training job** trains the model on a machine that exists only for a few minutes. Then the same model is served three ways, each with a different bill. A **real-time endpoint** answers one order at a time in milliseconds. A **serverless endpoint** does the same without a machine running all day. A **batch transform job** scores a whole file when nobody is waiting. All of it is started from Python files with the SageMaker Python SDK. The console is where you watch it happen.

Getting a model to answer once is the easy part. The class also covers what teams add before they trust an endpoint with real traffic. Those are cheaper training on spare capacity, the endpoint's own metrics, a public URL with a check in front, releasing a new model without dropping a request, adding machines when traffic grows, and what a failed release looks like.

**[Note]** Sections 1.1 to 1.3 happen in the browser, in **CloudShell**, with the region set to **N. Virginia (us-west-2)**. From section 1.4 on, the work happens in **VS Code on your laptop, connected to the `leviathan-lab` space**, in the `leviathan` folder from the previous class. Each new Python file goes in that folder, and each one is run from **VS Code's terminal**, which is a terminal on the space, with `python <file>.py`. Keep the console open in a browser tab beside VS Code to watch jobs and endpoints. Outputs below come from a real run, so your timings will differ a little. Section 3.4 also uses the console and a terminal on your laptop. Section 6 deletes everything that bills. Do not end the class without it.

**By the end you will be able to do all of this.**

1. Rebuild the whole platform in an empty account with one script, check it, and prove from inside the space that everything is reachable.
2. Start a training job from a Python file, read its metrics and logs, and run it again on spot capacity for less.
3. Deploy the model to a real-time endpoint from code and call it, and recognise a bad request that it silently accepts.
4. Read an endpoint's request, error and latency numbers in CloudWatch.
5. Put a public URL in front of the endpoint with API Gateway and Lambda, have it turn away malformed orders, and limit how fast it can be called.
6. Release a new model to a live endpoint with blue/green deployment and automatic rollback, and recognise a release that failed.
7. Add autoscaling to an endpoint and watch it add a machine under load.
8. Deploy the same model to a serverless endpoint, work around an SDK bug, move the public URL onto it, and measure its cold start.
9. Score a whole file with a batch transform job.
10. Choose between the three ways of serving for a given use.
11. Tear everything down in the right order.

**Prerequisites.** The previous class, so that every piece the script builds is familiar. AUC and the train and validation split.

## **1. Building the platform from nothing**

This class starts from an empty AWS account. We deleted everything in last class, so the VPC, the role, the bucket and the domain from the previous class are all gone. Companies meet the same situation when they open a fresh account for testing, or rebuild after a mistake. Clicking it all together again in the console would take most of the class.

Teams do not rebuild by hand. The platform is written down as code, so bringing it back is one command and gives the same result every time. This class uses two scripts for it. `check.sh` only reads. It lists every piece the class needs and says whether each one is there. `restore.sh` creates whatever is missing. It looks for each piece by name first and never makes a second copy, so running it again is safe.

| Piece | Why the class needs it |
|---|---|
| VPC, two private subnets, their route tables, internet gateway | The private network the space lives in |
| NAT gateway and a route to it from each subnet | Lets the space reach the internet, for VS Code and `pip` |
| S3 gateway endpoint and six interface endpoints | Private paths to S3 and to the AWS services the scripts call |
| Two security groups | Which machines may talk to the endpoints |
| `leviathan-sagemaker-role` | What the space and every job are allowed to do |
| Bucket, raw data and the four prepared files | What training and scoring read |
| Domain, user profile `instructor`, space `leviathan-lab` | Where VS Code connects |

### 1.1 Check what is there

Bucket names are shared by every AWS account in the world, so each of you needs your own. Something like `leviathan-` followed by your initials and today's date works. Both scripts take it as their only argument.

**[Note]** Browser, CloudShell.

1. Check the region at the top right of the console says **United States (N. Virginia)**. Open **CloudShell** with the terminal icon in the top bar, a little right of the search box, or the **CloudShell** button at the bottom left.
2. The first time, a **Welcome to AWS CloudShell** box opens over the terminal. Click **Close**. Wait for the `~ $` prompt.
3. Replace `YOUR-BUCKET-NAME` in the last line of the block below with your bucket name, copy the whole block and paste it into the terminal.
4. CloudShell asks before pasting more than one line. In the **Safe Paste for multiline text** box, click **Paste**. The orange button is **Cancel**, so do not just press Enter.
5. If the last line has not run by itself, press Enter. The block saves the script as `check.sh` and runs it. Next time, `bash check.sh YOUR-BUCKET-NAME` is enough.

![cs_safe_paste.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/804/original/cs_safe_paste.png?1791546649)

**Copy into CloudShell — replace YOUR-BUCKET-NAME in the last line first**

In [ ]:
cat > check.sh <<'EOF'
#!/bin/bash
# List every piece this class needs and say whether it is there. Only reads.
# Usage: bash check.sh YOUR-BUCKET-NAME
BUCKET=${1:?usage: bash check.sh YOUR-BUCKET-NAME}
export AWS_DEFAULT_REGION=us-west-2 AWS_REGION=us-west-2
# show NAME VALUE WANTED: MISSING if VALUE is empty, OK if its last word is WANTED, NOT-YET otherwise
show() {
  if [ -z "$2" ] || [ "$2" = "None" ]; then STATUS=MISSING
  elif [ -z "$3" ] || [ "${2##*[[:space:]]}" = "$3" ]; then STATUS=OK
  else STATUS=NOT-YET; fi
  printf '%-8s %-40s %s\n' "$STATUS" "$1" "$2"
}

VPC=$(aws ec2 describe-vpcs --filters Name=tag:Name,Values=leviathan-vpc --query 'Vpcs[0].VpcId' --output text)
show "VPC leviathan-vpc" "$VPC"
if [ "$VPC" = None ]; then
  # Everything below lives inside the VPC, so it cannot exist either.
  for NAME in "Internet gateway" "NAT gateway" "Endpoints" "Security groups"; do show "$NAME" ""; done
else
  IGW=$(aws ec2 describe-internet-gateways --filters Name=attachment.vpc-id,Values=$VPC --query 'InternetGateways[0].InternetGatewayId' --output text)
  show "Internet gateway" "$IGW"

  NAT=$(aws ec2 describe-nat-gateways --filter Name=vpc-id,Values=$VPC Name=state,Values=pending,available --query 'NatGateways[0].[NatGatewayId,State]' --output text)
  show "NAT gateway" "$NAT" available

  for SN in $(aws ec2 describe-subnets --filters Name=vpc-id,Values=$VPC --query 'Subnets[].SubnetId' --output text); do
    RT=$(aws ec2 describe-route-tables --filters Name=association.subnet-id,Values=$SN --query 'RouteTables[0].RouteTableId' --output text)
    ROUTE=$(aws ec2 describe-route-tables --route-table-ids $RT --query "RouteTables[0].Routes[?DestinationCidrBlock=='0.0.0.0/0'].[NatGatewayId,State]" --output text)
    show "Internet route in $RT" "$ROUTE" active
  done

  for SERVICE in s3 sagemaker.api sagemaker.runtime sts logs ssm ssmmessages; do
    STATE=$(aws ec2 describe-vpc-endpoints --filters Name=vpc-id,Values=$VPC Name=service-name,Values=com.amazonaws.us-west-2.$SERVICE \
      --query 'VpcEndpoints[?State!=`deleted` && State!=`deleting`]|[0].State' --output text)
    show "Endpoint $SERVICE" "$STATE" available
  done

  for GROUP in leviathan-studio-sg leviathan-endpoints-sg; do
    SG=$(aws ec2 describe-security-groups --filters Name=vpc-id,Values=$VPC Name=group-name,Values=$GROUP --query 'SecurityGroups[0].GroupId' --output text)
    show "Security group $GROUP" "$SG"
  done
fi

ROLE=$(aws iam get-role --role-name leviathan-sagemaker-role --query Role.RoleName --output text 2>/dev/null)
show "Role" "$ROLE"
RA=$(aws iam get-role-policy --role-name leviathan-sagemaker-role --policy-name leviathan-remote-access \
  --query 'PolicyDocument.Statement[0].Action' --output text 2>/dev/null)
show "  remote-access policy (StartSession)" "$RA"
aws s3api head-bucket --bucket $BUCKET > /dev/null 2>&1 && show "Bucket $BUCKET" "found" || show "Bucket $BUCKET" ""
for KEY in raw/returns.csv train/train.csv validation/validation.csv batch-input/batch.csv batch-labels/labels.csv; do
  aws s3api head-object --bucket $BUCKET --key $KEY > /dev/null 2>&1 && show "  $KEY" "found" || show "  $KEY" ""
done

DOMAIN=$(aws sagemaker list-domains --query "Domains[?DomainName=='leviathan-domain' && Status!='Deleting']|[0].[DomainId,Status]" --output text)
show "Domain leviathan-domain" "$DOMAIN" InService
if [ "$DOMAIN" != None ]; then
  ID=${DOMAIN%%[[:space:]]*}
  show "User profile instructor" "$(aws sagemaker describe-user-profile --domain-id $ID --user-profile-name instructor --query Status --output text 2>/dev/null)" InService
  show "Space leviathan-lab" "$(aws sagemaker describe-space --domain-id $ID --space-name leviathan-lab --query Status --output text 2>/dev/null)" InService
  SPACE_ARN=$(aws sagemaker describe-space --domain-id $ID --space-name leviathan-lab --query SpaceArn --output text 2>/dev/null)
  [ -n "$SPACE_ARN" ] && TAG=$(aws sagemaker list-tags --resource-arn $SPACE_ARN --query "Tags[?Key=='sagemaker:user-profile-arn'].Value" --output text)
  show "  owner tag (needed for remote access)" "${TAG##*/}" instructor
else
  show "User profile instructor" ""; show "Space leviathan-lab" ""
fi
EOF
bash check.sh YOUR-BUCKET-NAME


Output from the run, on the wiped account, with the bucket name `leviathan-class12-demo`.

```
MISSING  VPC leviathan-vpc                       None
MISSING  Internet gateway                         
MISSING  NAT gateway                              
MISSING  Endpoints                                
MISSING  Security groups                          
MISSING  Role                                     
MISSING  Bucket leviathan-class12-demo           
MISSING    raw/returns.csv                        
MISSING    train/train.csv                        
MISSING    validation/validation.csv              
MISSING    batch-input/batch.csv                  
MISSING    batch-labels/labels.csv                
MISSING  Domain leviathan-domain                 None
MISSING  User profile instructor                  
MISSING  Space leviathan-lab
```

Each line starts with one word. `OK` means the piece is there and ready. `MISSING` means it does not exist. `NOT-YET` means it exists but is not usable yet. Here every line says `MISSING`. When the VPC is missing, the script does not look for the gateways, endpoints and security groups that would live inside it, because none of them can exist.

### 1.2 Build it with restore.sh

`restore.sh` works top to bottom in the same order as the table.

- **Network.** The VPC with DNS turned on, an internet gateway, two private subnets in different zones, a route table for each, the S3 gateway endpoint and the two security groups. Then a regional NAT gateway, which the script waits for, a route from each subnet to it, and the six interface endpoints.
- **Role.** The role with the same three AWS policies and two inline policies as the previous class. The bucket policy is written every time the script runs, so it always names the bucket you passed in.
- **Data.** The bucket, the raw `returns.csv` downloaded from the course's file store, and the four prepared files. The script writes those with the same code as `prepare.py`, after installing pandas into a throwaway Python environment.
- **Studio.** The VPC-only domain using the role, the studio security group and both subnets, then the `instructor` user profile and the `leviathan-lab` space. The domain stops an idle space after 180 minutes, not the 60 of the previous class. In a test run of this class the 60-minute setting stopped the space while scripts were still running over the remote connection, because that kind of work does not count as use. The space is tagged with the profile that owns it, because the remote-access policy only lets a user connect to spaces carrying that tag. The console adds the tag by itself, but a space made from a script does not. The space gets an `ml.m5.large` machine, remote access turned on and the SageMaker Distribution image pinned at version 4.5.0. That is the version this class was recorded with, and the SDK bug in section 4 belongs to it.

Each line it prints starts with `created` or `found`, so a second run shows at a glance that nothing new was made.

**[Note]** CloudShell. Replace `YOUR-BUCKET-NAME` in the last line, paste the whole block, click **Paste** in the **Safe Paste** box and press Enter if needed. It takes several minutes, most of it waiting for the NAT gateway and the domain. Stay on the tab. CloudShell ends a session that gets no keyboard input for a while and shows **Your session has ended because no user interaction occurred** with a **Reconnect** button. If that happens, or the script stops half way for any other reason, click **Reconnect** and run `bash restore.sh YOUR-BUCKET-NAME` again. The saved scripts are still in the home folder, and the script carries on from where it stopped.

**Copy into CloudShell — replace YOUR-BUCKET-NAME in the last line first**

In [1]:
cat > restore.sh <<'EOF'
#!/bin/bash
# Bring the account to the state this class starts from: network, role, bucket and data, domain and space.
# It looks for each piece by name and creates only what is missing, so it is safe to run again.
# Usage: bash restore.sh YOUR-BUCKET-NAME
set -euo pipefail
BUCKET=${1:?usage: bash restore.sh YOUR-BUCKET-NAME}
export AWS_DEFAULT_REGION=us-west-2 AWS_REGION=us-west-2
DATA_URL="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/530/original/returns.csv?1791270327"
IMAGE="arn:aws:sagemaker:us-west-2:885854791233:image/sagemaker-distribution-cpu"
tag() { echo "ResourceType=$1,Tags=[{Key=Name,Value=$2}]"; }

# --- Network -------------------------------------------------------------------------------
VPC=$(aws ec2 describe-vpcs --filters Name=tag:Name,Values=leviathan-vpc --query 'Vpcs[0].VpcId' --output text)
if [ "$VPC" = None ]; then
  VPC=$(aws ec2 create-vpc --cidr-block 10.0.0.0/16 --tag-specifications "$(tag vpc leviathan-vpc)" --query Vpc.VpcId --output text)
  aws ec2 wait vpc-available --vpc-ids $VPC
  aws ec2 modify-vpc-attribute --vpc-id $VPC --enable-dns-support '{"Value":true}'
  aws ec2 modify-vpc-attribute --vpc-id $VPC --enable-dns-hostnames '{"Value":true}'
  echo "created VPC $VPC"
else echo "found   VPC $VPC"; fi

IGW=$(aws ec2 describe-internet-gateways --filters Name=attachment.vpc-id,Values=$VPC --query 'InternetGateways[0].InternetGatewayId' --output text)
if [ "$IGW" = None ]; then
  IGW=$(aws ec2 create-internet-gateway --tag-specifications "$(tag internet-gateway leviathan-igw)" --query InternetGateway.InternetGatewayId --output text)
  aws ec2 attach-internet-gateway --internet-gateway-id $IGW --vpc-id $VPC
  echo "created internet gateway $IGW"
else echo "found   internet gateway $IGW"; fi

# Use the VPC's own subnets if it has any (the previous class made two). Otherwise make two private ones.
SUBNETS=$(aws ec2 describe-subnets --filters Name=vpc-id,Values=$VPC --query 'Subnets[].SubnetId' --output text)
if [ -z "$SUBNETS" ]; then
  i=1
  for AZ_CIDR in us-west-2b:10.0.128.0/20 us-west-2c:10.0.144.0/20; do
    AZ=${AZ_CIDR%%:*}; CIDR=${AZ_CIDR#*:}
    SN=$(aws ec2 create-subnet --vpc-id $VPC --cidr-block $CIDR --availability-zone $AZ \
         --tag-specifications "$(tag subnet leviathan-subnet-private$i-$AZ)" --query Subnet.SubnetId --output text)
    echo "created subnet $SN in $AZ"
    SUBNETS="$SUBNETS $SN"; i=$((i + 1))
  done
else echo "found   subnets" $SUBNETS; fi
RTS=""; i=1
for SN in $SUBNETS; do
  RT=$(aws ec2 describe-route-tables --filters Name=association.subnet-id,Values=$SN --query 'RouteTables[0].RouteTableId' --output text)
  if [ "$RT" = None ]; then
    AZ=$(aws ec2 describe-subnets --subnet-ids $SN --query 'Subnets[0].AvailabilityZone' --output text)
    RT=$(aws ec2 create-route-table --vpc-id $VPC --tag-specifications "$(tag route-table leviathan-rtb-private$i-$AZ)" \
         --query RouteTable.RouteTableId --output text)
    aws ec2 associate-route-table --route-table-id $RT --subnet-id $SN > /dev/null
    echo "created route table $RT for $SN"
  else echo "found   route table $RT for $SN"; fi
  RTS="$RTS $RT"; i=$((i + 1))
done

S3EP=$(aws ec2 describe-vpc-endpoints --filters Name=vpc-id,Values=$VPC Name=service-name,Values=com.amazonaws.us-west-2.s3 \
       --query 'VpcEndpoints[?State!=`deleted` && State!=`deleting`]|[0].VpcEndpointId' --output text)
if [ "$S3EP" = None ]; then
  S3EP=$(aws ec2 create-vpc-endpoint --vpc-id $VPC --vpc-endpoint-type Gateway --service-name com.amazonaws.us-west-2.s3 \
         --route-table-ids $RTS --tag-specifications "$(tag vpc-endpoint leviathan-vpce-s3)" --query VpcEndpoint.VpcEndpointId --output text)
  echo "created S3 gateway endpoint $S3EP"
else echo "found   S3 gateway endpoint $S3EP"; fi

sg() { aws ec2 describe-security-groups --filters Name=vpc-id,Values=$VPC Name=group-name,Values=$1 --query 'SecurityGroups[0].GroupId' --output text; }
STUDIO=$(sg leviathan-studio-sg)
if [ "$STUDIO" = None ]; then
  STUDIO=$(aws ec2 create-security-group --vpc-id $VPC --group-name leviathan-studio-sg --description "Studio domain and its spaces" --query GroupId --output text)
  aws ec2 authorize-security-group-ingress --group-id $STUDIO --ip-permissions "IpProtocol=tcp,FromPort=0,ToPort=65535,UserIdGroupPairs=[{GroupId=$STUDIO}]" > /dev/null
  echo "created security group leviathan-studio-sg $STUDIO"
else echo "found   security group leviathan-studio-sg $STUDIO"; fi
ENDP=$(sg leviathan-endpoints-sg)
if [ "$ENDP" = None ]; then
  ENDP=$(aws ec2 create-security-group --vpc-id $VPC --group-name leviathan-endpoints-sg --description "Interface endpoints, HTTPS from Studio" --query GroupId --output text)
  aws ec2 authorize-security-group-ingress --group-id $ENDP --ip-permissions "IpProtocol=tcp,FromPort=443,ToPort=443,UserIdGroupPairs=[{GroupId=$STUDIO}]" > /dev/null
  echo "created security group leviathan-endpoints-sg $ENDP"
else echo "found   security group leviathan-endpoints-sg $ENDP"; fi

NAT=$(aws ec2 describe-nat-gateways --filter Name=vpc-id,Values=$VPC Name=state,Values=pending,available --query 'NatGateways[0].NatGatewayId' --output text)
if [ "$NAT" = None ]; then
  NAT=$(aws ec2 create-nat-gateway --vpc-id $VPC --availability-mode regional \
        --tag-specifications "$(tag natgateway leviathan-regional-nat)" --query NatGateway.NatGatewayId --output text)
  echo "created NAT gateway $NAT, waiting for it (two to three minutes)"
else echo "found   NAT gateway $NAT"; fi
aws ec2 wait nat-gateway-available --nat-gateway-ids $NAT

for RT in $RTS; do
  NOW=$(aws ec2 describe-route-tables --route-table-ids $RT --query "RouteTables[0].Routes[?DestinationCidrBlock=='0.0.0.0/0'].[NatGatewayId,State]" --output text)
  if [ -z "$NOW" ]; then
    aws ec2 create-route --route-table-id $RT --destination-cidr-block 0.0.0.0/0 --nat-gateway-id $NAT > /dev/null
    echo "created route in $RT to $NAT"
  elif [ "$NOW" != "$(printf '%s\tactive' $NAT)" ]; then
    aws ec2 replace-route --route-table-id $RT --destination-cidr-block 0.0.0.0/0 --nat-gateway-id $NAT
    echo "pointed route in $RT at $NAT (it was: $NOW)"
  else echo "found   route in $RT to $NAT"; fi
done

for SERVICE in sagemaker.api sagemaker.runtime sts logs ssm ssmmessages; do
  EP=$(aws ec2 describe-vpc-endpoints --filters Name=vpc-id,Values=$VPC Name=service-name,Values=com.amazonaws.us-west-2.$SERVICE \
       --query 'VpcEndpoints[?State!=`deleted` && State!=`deleting`]|[0].VpcEndpointId' --output text)
  if [ "$EP" = None ]; then
    EP=$(aws ec2 create-vpc-endpoint --vpc-id $VPC --vpc-endpoint-type Interface \
         --service-name com.amazonaws.us-west-2.$SERVICE --subnet-ids $SUBNETS \
         --security-group-ids $ENDP --private-dns-enabled \
         --tag-specifications "$(tag vpc-endpoint leviathan-$SERVICE)" --query VpcEndpoint.VpcEndpointId --output text)
    echo "created endpoint $SERVICE $EP"
  else echo "found   endpoint $SERVICE $EP"; fi
done

# --- Role ----------------------------------------------------------------------------------
ROLE=leviathan-sagemaker-role
if ! aws iam get-role --role-name $ROLE > /dev/null 2>&1; then
  aws iam create-role --role-name $ROLE --assume-role-policy-document \
    '{"Version":"2012-10-17","Statement":[{"Effect":"Allow","Principal":{"Service":"sagemaker.amazonaws.com"},"Action":"sts:AssumeRole"}]}' > /dev/null
  for P in AmazonSageMakerFullAccess AmazonSageMakerCanvasFullAccess AmazonSageMakerCanvasDataPrepFullAccess; do
    aws iam attach-role-policy --role-name $ROLE --policy-arn arn:aws:iam::aws:policy/$P
  done
  echo "created role $ROLE, waiting 15 s for IAM to catch up"; sleep 15
else echo "found   role $ROLE"; fi
# Always rewritten too. A role left over from an earlier run may lack it, and without
# sagemaker:StartSession the space refuses VS Code with "Remote access denied".
aws iam put-role-policy --role-name $ROLE --policy-name leviathan-remote-access --policy-document \
  '{"Version":"2012-10-17","Statement":[{"Effect":"Allow","Action":["sagemaker:StartSession"],"Resource":"arn:*:sagemaker:*:*:space/${sagemaker:DomainId}/*","Condition":{"ArnLike":{"sagemaker:ResourceTag/sagemaker:user-profile-arn":"arn:aws:sagemaker:*:*:user-profile/${sagemaker:DomainId}/${sagemaker:UserProfileName}"}}}]}'
# Always rewritten, so the role names the bucket you passed in.
aws iam put-role-policy --role-name $ROLE --policy-name leviathan-bucket-access --policy-document \
  "{\"Version\":\"2012-10-17\",\"Statement\":[{\"Effect\":\"Allow\",\"Action\":[\"s3:ListBucket\",\"s3:GetBucketLocation\"],\"Resource\":\"arn:aws:s3:::$BUCKET\"},{\"Effect\":\"Allow\",\"Action\":[\"s3:GetObject\",\"s3:PutObject\",\"s3:DeleteObject\"],\"Resource\":\"arn:aws:s3:::$BUCKET/*\"}]}"
ROLE_ARN=$(aws iam get-role --role-name $ROLE --query Role.Arn --output text)

# --- Bucket and data -----------------------------------------------------------------------
if ! aws s3api head-bucket --bucket $BUCKET > /dev/null 2>&1; then
  aws s3 mb s3://$BUCKET > /dev/null; echo "created bucket $BUCKET"
else echo "found   bucket $BUCKET"; fi
if ! aws s3api head-object --bucket $BUCKET --key raw/returns.csv > /dev/null 2>&1; then
  curl -sf "$DATA_URL" | aws s3 cp - s3://$BUCKET/raw/returns.csv; echo "uploaded raw/returns.csv"
else echo "found   raw/returns.csv"; fi
if ! aws s3api head-object --bucket $BUCKET --key train/train.csv > /dev/null 2>&1; then
  echo "preparing the four files, as prepare.py did in the previous class"
  python3 -m venv /tmp/restore-venv && /tmp/restore-venv/bin/pip install -q pandas boto3
  BUCKET=$BUCKET /tmp/restore-venv/bin/python - <<'PY'
import os, boto3, pandas as pd
BUCKET = os.environ["BUCKET"]
s3 = boto3.client("s3")
raw = pd.read_csv(s3.get_object(Bucket=BUCKET, Key="raw/returns.csv")["Body"])
features = raw.drop(columns=["order_id", "order_date", "refund_amount", "returned"])
features = pd.get_dummies(features, columns=["category", "payment_method"], dtype=int)
data = pd.concat([raw["returned"], features], axis=1).sample(frac=1, random_state=0)
n = len(data)
train, validation, batch = data.iloc[: int(n * 0.7)], data.iloc[int(n * 0.7): int(n * 0.85)], data.iloc[int(n * 0.85):]
put = lambda df, key, header=False: s3.put_object(Bucket=BUCKET, Key=key, Body=df.to_csv(header=header, index=False))
put(train, "train/train.csv"); put(validation, "validation/validation.csv")
put(batch.drop(columns=["returned"]), "batch-input/batch.csv"); put(batch[["returned"]], "batch-labels/labels.csv", header=True)
print("wrote train", train.shape, "validation", validation.shape, "batch", batch.shape)
PY
else echo "found   train/train.csv and the other prepared files"; fi

# --- Domain, user profile and space ----------------------------------------------------------
# Idle shutdown is 180 minutes: work over the remote connection does not count as use, and this class runs long jobs.
DOMAIN=$(aws sagemaker list-domains --query "Domains[?DomainName=='leviathan-domain' && Status!='Deleting']|[0].DomainId" --output text)
if [ "$DOMAIN" = None ]; then
  DOMAIN=$(aws sagemaker create-domain --domain-name leviathan-domain --auth-mode IAM --vpc-id $VPC --subnet-ids $SUBNETS \
    --app-network-access-type VpcOnly \
    --default-user-settings "{\"ExecutionRole\":\"$ROLE_ARN\",\"SecurityGroups\":[\"$STUDIO\"],\"JupyterLabAppSettings\":{\"AppLifecycleManagement\":{\"IdleSettings\":{\"LifecycleManagement\":\"ENABLED\",\"IdleTimeoutInMinutes\":180}}}}" \
    --default-space-settings "{\"ExecutionRole\":\"$ROLE_ARN\",\"SecurityGroups\":[\"$STUDIO\"]}" \
    --query DomainArn --output text | cut -d/ -f2)
  echo "created domain $DOMAIN, waiting for it"
else echo "found   domain $DOMAIN"; fi
until [ "$(aws sagemaker describe-domain --domain-id $DOMAIN --query Status --output text)" = InService ]; do sleep 10; done

if ! aws sagemaker describe-user-profile --domain-id $DOMAIN --user-profile-name instructor > /dev/null 2>&1; then
  aws sagemaker create-user-profile --domain-id $DOMAIN --user-profile-name instructor > /dev/null
  echo "created user profile instructor"
else echo "found   user profile instructor"; fi
until [ "$(aws sagemaker describe-user-profile --domain-id $DOMAIN --user-profile-name instructor --query Status --output text)" = InService ]; do sleep 5; done

if ! aws sagemaker describe-space --domain-id $DOMAIN --space-name leviathan-lab > /dev/null 2>&1; then
  aws sagemaker create-space --domain-id $DOMAIN --space-name leviathan-lab \
    --ownership-settings OwnerUserProfileName=instructor --space-sharing-settings SharingType=Private \
    --space-settings "{\"AppType\":\"JupyterLab\",\"RemoteAccess\":\"ENABLED\",\"SpaceStorageSettings\":{\"EbsStorageSettings\":{\"EbsVolumeSizeInGb\":5}},\"JupyterLabAppSettings\":{\"DefaultResourceSpec\":{\"InstanceType\":\"ml.m5.large\",\"SageMakerImageArn\":\"$IMAGE\",\"SageMakerImageVersionAlias\":\"4.5.0\"}}}" > /dev/null
  echo "created space leviathan-lab (ml.m5.large, remote access on, stopped)"
else echo "found   space leviathan-lab"; fi
# The remote-access policy only allows spaces tagged with their owner's profile. The console adds
# this tag, create-space does not, and without it VS Code gets "Remote access denied".
aws sagemaker add-tags --resource-arn $(aws sagemaker describe-space --domain-id $DOMAIN --space-name leviathan-lab --query SpaceArn --output text) \
  --tags Key=sagemaker:user-profile-arn,Value=$(aws sagemaker describe-user-profile --domain-id $DOMAIN --user-profile-name instructor --query UserProfileArn --output text) > /dev/null
echo "done. Interface endpoints can take another minute to become available. Run check.sh."
EOF
bash restore.sh YOUR-BUCKET-NAME


SyntaxError: invalid decimal literal (1947304865.py, line 34)

Output from the run.

```
created VPC vpc-05f9cce6e58cfc038
created internet gateway igw-0ae32157d7f27c3c7
created subnet subnet-0834f7773e5b800a2 in us-west-2b
created subnet subnet-09d91d00bca605ef7 in us-west-2c
created route table rtb-0fa115b5833df23f3 for subnet-0834f7773e5b800a2
created route table rtb-09f09e24d9345a456 for subnet-09d91d00bca605ef7
created S3 gateway endpoint vpce-0f2e751c7a05a7d64
created security group leviathan-studio-sg sg-0b3d6ffc929278bba
created security group leviathan-endpoints-sg sg-08f5886dfe1556808
created NAT gateway nat-1fe797d2c361271d8, waiting for it (two to three minutes)
created route in rtb-0fa115b5833df23f3 to nat-1fe797d2c361271d8
created route in rtb-09f09e24d9345a456 to nat-1fe797d2c361271d8
created endpoint sagemaker.api vpce-0bee2ea60bf962776
created endpoint sagemaker.runtime vpce-0ed752fa111dce4bd
created endpoint sts vpce-026517906b6525269
created endpoint logs vpce-094f5c8c2ee28f3c7
created endpoint ssm vpce-002e4f26307698769
created endpoint ssmmessages vpce-0931fc1675f3771da
created role leviathan-sagemaker-role, waiting 15 s for IAM to catch up
created bucket leviathan-class12-demo
uploaded raw/returns.csv
preparing the four files, as prepare.py did in the previous class
wrote train (8400, 19) validation (1800, 19) batch (1800, 19)
created domain d-whlcql7pqzvp, waiting for it
created user profile instructor
created space leviathan-lab (ml.m5.large, remote access on, stopped)
done. Interface endpoints can take another minute to become available. Run check.sh.
```

Every line says `created`, which is what an empty account should give. A second run, in CloudShell, printed `found` on all 24 lines and created nothing, so running it again is safe. The prepared files came out with 8,400, 1,800 and 1,800 rows, the same split as `prepare.py` gave in the previous class, and the first line of `train.csv` is identical to the one it printed then.

**Resume or rerun the saved restore script — CloudShell. Replace YOUR-BUCKET-NAME.**

In [ ]:
bash restore.sh YOUR-BUCKET-NAME

### 1.3 Check again

**[Note]** CloudShell. Run `bash check.sh YOUR-BUCKET-NAME`. If any endpoint still says `NOT-YET`, wait a minute and run it again. Do not start the space until every line says `OK`.

**Copy into CloudShell**

In [ ]:
bash check.sh YOUR-BUCKET-NAME


Output from the run, straight after `restore.sh` finished.

```
OK       VPC leviathan-vpc                       vpc-05f9cce6e58cfc038
OK       Internet gateway                         igw-0ae32157d7f27c3c7
OK       NAT gateway                              nat-1fe797d2c361271d8	available
OK       Internet route in rtb-0fa115b5833df23f3  nat-1fe797d2c361271d8	active
OK       Internet route in rtb-09f09e24d9345a456  nat-1fe797d2c361271d8	active
OK       Endpoint s3                              available
OK       Endpoint sagemaker.api                   available
OK       Endpoint sagemaker.runtime               available
OK       Endpoint sts                             available
OK       Endpoint logs                            available
OK       Endpoint ssm                             available
OK       Endpoint ssmmessages                     available
OK       Security group leviathan-studio-sg         sg-0b3d6ffc929278bba
OK       Security group leviathan-endpoints-sg      sg-08f5886dfe1556808
OK       Role                                     leviathan-sagemaker-role
OK       Bucket leviathan-class12-demo           found
OK         raw/returns.csv                        found
OK         train/train.csv                        found
OK         validation/validation.csv              found
OK         batch-input/batch.csv                  found
OK         batch-labels/labels.csv                found
OK       Domain leviathan-domain                 d-whlcql7pqzvp	InService
OK       User profile instructor                  InService
OK       Space leviathan-lab                        InService
```

Every line says `OK`, and the two route lines name the new NAT gateway and say `active`.

### 1.4 Into VS Code, and a check from inside

**[Note]** Browser, then laptop.

1. Type `SageMaker AI` in the console search bar and open it. In the menu on the left, under **Applications and IDEs**, click **SageMaker Studio**.

**AWS console search**

In [ ]:
SageMaker AI

2. Pick the `instructor` profile and click **Open Studio**.

**User profile**

In [ ]:
instructor

3. Open **JupyterLab**, open `leviathan-lab` and click **Run space**. `restore.sh` already set its machine, its image and the **Remote access** switch.

**Space name**

In [ ]:
leviathan-lab

4. When it is running, click **Open in VS Code**. Right after the space starts, the connection can fail for a few minutes with "is not connected, please wait before retrying". Wait and try again.
5. The space is new, so the `leviathan` folder from the previous class is gone with the old disk. Open Folder only lists folders that exist, so make it first. In the remote window, open a terminal with **Terminal → New Terminal**. It starts in `/home/sagemaker-user`. Run `mkdir -p leviathan`. Then choose **File → Open Folder**, pick `leviathan` and click **OK**. The window reloads into the folder. Open the terminal again, and it now starts inside `leviathan`.

**Create the project folder — VS Code remote terminal**

In [ ]:
mkdir -p leviathan

**Folder to open — File → Open Folder**

In [ ]:
/home/sagemaker-user/leviathan

Every file in this class reads the bucket name from one place.

**[Note]** VS Code, new file `leviathan/config.py`. Change `BUCKET` to your own bucket name. Nothing to run.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
config.py

**File contents — paste into the new file and save**

In [ ]:
"""Settings every script in this folder shares. Change BUCKET to your own bucket."""
BUCKET = "leviathan-demo-1007"
REGION = "us-west-2"

`check.sh` looked at the account from outside. The last check runs inside the space and calls each service the class uses. It asks STS which role it is running as, lists training jobs through `sagemaker.api`, lists your bucket through the S3 endpoint, and looks up the address of each service that has an interface endpoint. An address starting with `10.0.` is inside your VPC, which proves private DNS is sending that service to the endpoint. Last, it fetches a page from `pypi.org`, which only works through the NAT gateway.

**[Note]** VS Code, new file `leviathan/check_space.py`. Run it with `python check_space.py`. Every line should say `OK`.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
check_space.py

**File contents — paste into the new file and save**

In [ ]:
"""Check, from inside the space, that everything this class calls is reachable."""
import socket
import urllib.request

import boto3

from config import BUCKET, REGION


def check(name, step):
    try:
        print(f"{'OK':8} {name:38} {step()}")
    except Exception as e:
        print(f"{'FAILED':8} {name:38} {type(e).__name__}: {str(e)[:90]}")


check("role, through sts", lambda: boto3.client("sts").get_caller_identity()["Arn"].split("/")[1])
check("sagemaker.api", lambda: f"answered, HTTP {boto3.client('sagemaker').list_training_jobs(MaxResults=1)['ResponseMetadata']['HTTPStatusCode']}")
check("bucket, through s3", lambda: sorted(o["Key"] for o in boto3.client("s3").list_objects_v2(Bucket=BUCKET)["Contents"]))
for host in ["api.sagemaker", "runtime.sagemaker", "sts", "logs"]:
    name = f"{host}.{REGION}.amazonaws.com"
    check(f"private address for {host}", lambda name=name: socket.gethostbyname(name))
check("internet, through the NAT", lambda: f"pypi.org answered {urllib.request.urlopen('https://pypi.org/simple/', timeout=10).status}")

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python check_space.py

Output from the run, on the space that `restore.sh` built.

```
OK       role, through sts                      leviathan-sagemaker-role
OK       sagemaker.api                          answered, HTTP 200
OK       bucket, through s3                     ['batch-input/batch.csv', 'batch-labels/labels.csv', 'raw/returns.csv', 'train/train.csv', 'validation/validation.csv']
OK       private address for api.sagemaker      10.0.147.82
OK       private address for runtime.sagemaker  10.0.136.135
OK       private address for sts                10.0.135.23
OK       private address for logs               10.0.134.23
OK       internet, through the NAT              pypi.org answered 200
```

Every line says `OK`. The role is `leviathan-sagemaker-role`, so the space runs as the role `restore.sh` made. The bucket listing shows the raw file and the four prepared files, which is everything the training job in section 2 reads. All four service names resolve to `10.0.` addresses, which are inside your VPC, so their traffic goes through the endpoints and not the internet. And `pypi.org` answered, so the NAT gateway works. If any line says `FAILED`, go back to `check.sh` first. A `FAILED` on the internet line points at the NAT gateway or a private route, so look at those lines of `check.sh` first.

## **2. A training job from code**

A **training job** is one request to SageMaker. It says which algorithm to run, which S3 prefixes to read, what machine to rent, and where in S3 to put the model. SageMaker starts the machine, downloads the data onto it, runs the container and uploads the result. Then it deletes the machine. You pay only for the seconds in between.

The **SageMaker Python SDK** is the library teams use to send that request from code. The space has version 3, which comes in a few parts. `sagemaker.train` holds `ModelTrainer`, the class that runs training jobs. `sagemaker.core` holds the building blocks, such as `image_uris`, which looks up the address of AWS's ready-made containers, and the small config objects that describe compute, data and output.

Each part of the file maps to one decision.

- `image_uris.retrieve("xgboost", REGION, "1.7-1")` gives the address of AWS's XGBoost container, version 1.7.
- `get_execution_role()` returns the role the space runs as, `leviathan-sagemaker-role`. The training job runs as that role too, which is why it can read your bucket.
- `Compute` is the machine. `ml.m5.large` with 5 GB of disk is plenty for 8,400 rows.
- `hyperparameters` are XGBoost's settings. `binary:logistic` makes the model output a probability, and `eval_metric` auc makes it report AUC as it trains.
- `InputData` is a **channel**, a name plus an S3 prefix. The XGBoost container looks for one called `train` and an optional one called `validation`.

**[Note]** VS Code, new file `leviathan/train.py`. Run it with `python train.py`. It waits until the job finishes, about three and a half minutes. While it waits, switch to the console tab, type `SageMaker AI` in the search bar at the top and open it. In the menu on the left, under **Model training & customization**, click **Training & tuning jobs**. If the group is folded, click its name first to open it. The **Training jobs** list opens with the newest job at the top. Click its name, which starts with `leviathan-xgb-`. On the job page, under **Job settings**, click **View history** below **Status** to show the steps the job has gone through so far.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
train.py

**File contents — paste into the new file and save**

In [ ]:
"""Start a SageMaker training job with the built-in XGBoost container."""
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core.training.configs import Compute, InputData, OutputDataConfig
from sagemaker.train import ModelTrainer

from config import BUCKET, REGION

image = image_uris.retrieve("xgboost", REGION, "1.7-1")
print("image:", image)

trainer = ModelTrainer(
    training_image=image,
    role=get_execution_role(),
    base_job_name="leviathan-xgb",
    compute=Compute(instance_type="ml.m5.large", instance_count=1, volume_size_in_gb=5),
    hyperparameters={
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "num_round": 200,
        "eta": 0.1,
        "max_depth": 4,
    },
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{BUCKET}/output/"),
)

trainer.train(
    input_data_config=[
        InputData(channel_name="train", data_source=f"s3://{BUCKET}/train/", content_type="text/csv"),
        InputData(channel_name="validation", data_source=f"s3://{BUCKET}/validation/", content_type="text/csv"),
    ],
    wait=True,
    logs=False,
)

job = trainer._latest_training_job  # the TrainingJob the SDK just ran
print("job:", job.training_job_name, job.training_job_status)
print("billable seconds:", job.billable_time_in_seconds)
print("final metrics:", {m.metric_name: round(m.value, 3) for m in job.final_metric_data_list})
print("model:", job.model_artifacts.s3_model_artifacts)

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python train.py

Output from the run.

```
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
[10/07/26 07:10:50] INFO     Ignoring unnecessary instance type: None.                                                               image_uris.py:535
image: 683313688378.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1
[10/07/26 07:10:51] INFO     SageMaker session not provided. Using default Session.                                                     defaults.py:61
                    INFO     StoppingCondition not provided. Using default:                                                            defaults.py:128
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None max_pending_time_in_seconds=None                               
                    INFO     OutputDataConfig compression type not provided. Using default:                                            defaults.py:165
                             GZIP                                                                                                                     
                    INFO     Training image URI: 683313688378.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1             model_trainer.py:558
                    INFO     SageMaker Python SDK will collect telemetry to help us better understand our user's needs,       telemetry_logging.py:322
                             diagnose issues, and deliver additional features.                                                                        
                             To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config.                            
                             For more information, refer to                                                                                           
                             https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the                         
                             -sagemaker-python-sdk.                                                                                                   
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
                    INFO     Creating training_job resource.                                                                        resources.py:31238
                    WARNING  No region provided. Using default region.                                                                    utils.py:361
[10/07/26 07:14:18] INFO     Final Resource Status: Completed                                                                       resources.py:31590
job: leviathan-xgb-20261007071051 Completed
billable seconds: 134
final metrics: {'validation:auc': 0.708, 'train:auc': 0.85}
model: s3://leviathan-demo-1007/output/leviathan-xgb-20261007071051/output/model.tar.gz
```

The SDK prints a few lines of its own. They say which defaults it filled in, such as a one-hour time limit, and that it collects usage data unless you turn that off. The last four lines are from the script.

The job name is `leviathan-xgb` plus the time it started. Names like that are what teams use, because **training jobs can never be deleted or renamed**. Every job stays in the list with its settings and logs for as long as the account exists.

To see the list in the console:

1. Type `SageMaker AI` in the search bar at the top of the console and open it.

**AWS console search**

In [ ]:
SageMaker AI

2. In the menu on the left, under **Model training & customization**, click **Training & tuning jobs**. If the group is folded, click its name first to open it.
3. The **Training jobs** list opens with the newest job at the top. The **Job status** column shows where each job is.

![tj_list.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/995/original/tj_list.png?1791359541)

The list has jobs in it that you did not start today. They are runs from preparing these classes. The failed one ran with a role that had no permission for its bucket, `leviathan-x15284`. Its failure reason looks like this.

```
ClientError: Data download failed:Failed to download data. AccessDenied (403):
User: arn:aws:sts::111122223333:assumed-role/AmazonSageMaker-ExecutionRole-.../SageMaker
is not authorized to perform: s3:GetObject on resource:
"arn:aws:s3:::leviathan-x15284/train/train.csv" because no identity-based policy allows the s3:GetObject action
```

The role could see `train.csv` but was not allowed to download it. The job failed after 101 seconds, and all 101 seconds were billed. Nothing checked the bucket before the job rented its machine. The inline bucket policy you gave the role in the previous class is what prevents this.

### 2.1 Reading the result

The job billed 134 seconds. At 0.115 USD an hour, that is less than half a US cent. To read its log:

1. In the **Training jobs** list, click the job's name to open its page.
2. Scroll down to the **Monitor** box and click **View logs**. CloudWatch opens in a new tab on the log group `/aws/sagemaker/TrainingJobs`, filtered to this job.
3. Click the log stream, whose name is the job name followed by `/algo-1-` and a number.

These are the lines worth reading.

```
Train matrix has 8400 rows and 18 columns
Validation matrix has 1800 rows
[0]   train-auc:0.70359   validation-auc:0.67208
[199] train-auc:0.84989   validation-auc:0.70779
```

The row counts match what `prepare.py` wrote, so the format was read correctly. The final validation AUC is 0.708, close to the 0.706 from the quick model in `explore.py` last class. Two different tools agreeing is a good sign that nothing went wrong in between. The training AUC is higher, at 0.850, which means the model fits the rows it saw better than new ones. That gap is normal, and the validation number is the one to trust.

The rest of the job page is worth a look too. Go back to the browser tab with the job's page.

1. Scroll past **Job settings** to **Algorithm**. It shows the training image and the machine, `ml.m5.large`, exactly as `train.py` set them. The input channels follow further down.

![tj_detail.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/994/original/tj_detail.png?1791359538)

2. Scroll on to the **Monitor** box. Its two charts show the final `train:auc` and `validation:auc`, and the **Output** box below it gives the S3 path of `model.tar.gz`.

![tj_monitor.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/996/original/tj_monitor.png?1791359542)

3. Scroll back to the top. In **Job settings**, click **View history** under **Status**.

![tj_history.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/580/original/tj_history.png?1791466266)

**View history** lists each step with its start and end time. Starting is SageMaker finding a machine. Downloading is fetching the container image and your data. Training is the part your hyperparameters control, and Uploading is writing `model.tar.gz` to S3. In this picture, from a later run of the same file, Starting took 42 seconds and Downloading 106, while Training took 10. For a small job like this one, most of the time goes on the steps around training.

The job page shows everything the code set. The one `model.tar.gz` file in its **Output** box is everything the next sections deploy.

### 2.2 Paying less with spot capacity

A training job does not need a machine at an exact moment. It can wait a few minutes for one. AWS sells its spare machines at a large discount as **spot** capacity, and SageMaker's name for using it is **managed spot training**. The catch is that AWS can take a spot machine back at any time. SageMaker then waits for another one and starts the job again.

Two settings deal with that. `max_wait_time_in_seconds` limits how long the job may take in total, waiting included, and it has to be at least `max_runtime_in_seconds`. `checkpoint_config` names an S3 prefix where the container saves its progress. A job that gets restarted picks up from the last save, instead of from the first round.

The file also adds **tags**. A tag is a key and a value attached to an AWS resource. Teams tag everything with at least the project name, so the bill can be split by project later in **Billing and Cost Management**. Everything else is the same as `train.py`, with the same container, data and hyperparameters.

**[Note]** VS Code, new file `leviathan/train_spot.py`. Run it with `python train_spot.py`. It takes about three minutes. When it finishes, open it in the console the same way as before, from **Model training & customization → Training & tuning jobs**, and show **Managed spot training savings** under **Job settings**.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
train_spot.py

**File contents — paste into the new file and save**

In [ ]:
"""The same training job on spare (spot) capacity, with checkpoints and cost tags."""
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core.training.configs import (CheckpointConfig, Compute, InputData,
                                             OutputDataConfig, StoppingCondition, Tag)
from sagemaker.train import ModelTrainer

from config import BUCKET, REGION

trainer = ModelTrainer(
    training_image=image_uris.retrieve("xgboost", REGION, "1.7-1"),
    role=get_execution_role(),
    base_job_name="leviathan-xgb-spot",
    compute=Compute(instance_type="ml.m5.large", instance_count=1, volume_size_in_gb=5,
                    enable_managed_spot_training=True),
    # Spot needs a second limit: how long to wait for spare capacity plus the training itself.
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600, max_wait_time_in_seconds=7200),
    # If AWS takes the machine back, the job restarts from the last checkpoint, not from zero.
    checkpoint_config=CheckpointConfig(s3_uri=f"s3://{BUCKET}/checkpoints/leviathan-xgb-spot/"),
    tags=[Tag(key="project", value="leviathan"), Tag(key="stage", value="experiment")],
    hyperparameters={
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "num_round": 200,
        "eta": 0.1,
        "max_depth": 4,
    },
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{BUCKET}/output/"),
)

trainer.train(
    input_data_config=[
        InputData(channel_name="train", data_source=f"s3://{BUCKET}/train/", content_type="text/csv"),
        InputData(channel_name="validation", data_source=f"s3://{BUCKET}/validation/", content_type="text/csv"),
    ],
    wait=True,
    logs=False,
)

job = trainer._latest_training_job
trained, billed = job.training_time_in_seconds, job.billable_time_in_seconds
print("job:", job.training_job_name, job.training_job_status)
print("training seconds:", trained, " billable seconds:", billed)
print(f"saving from spot: {100 * (1 - billed / trained):.0f}%")
print("final metrics:", {m.metric_name: round(m.value, 3) for m in job.final_metric_data_list})
print("model:", job.model_artifacts.s3_model_artifacts)

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python train_spot.py

Output from the run, without the SDK's own log lines. This file and the new files in section 3 were run on a later day, from a laptop using the same role, against a bucket with a different name.

```
job: leviathan-xgb-spot-20261008164917 Completed
training seconds: 124  billable seconds: 48
saving from spot: 61%
final metrics: {'validation:auc': 0.708, 'train:auc': 0.85}
model: s3://leviathan-hehe/output/leviathan-xgb-spot-20261008164917/output/model.tar.gz
```

The model came out the same, 0.850 and 0.708, because nothing about the training changed. The job trained for 124 seconds and was billed for 48, a saving of 61%. The job page shows the same figure.

1. Go back to **Model training & customization → Training & tuning jobs**. The spot job is at the top, named `leviathan-xgb-spot-` plus the time.
2. Click its name. **Job settings** shows **Training time**, **Billable time** and **Managed spot training savings** side by side, and **Algorithm** below shows **Managed spot training** as **Enabled**.

![tj_spot_detail.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/581/original/tj_spot_detail.png?1791466268)

The checkpoint prefix in the bucket now holds one file, `xgboost-checkpoint_0.json`. AWS did not take the machine back during this run, so the checkpoint was never needed. Spot is a sensible default for training jobs that can wait, and many teams turn it on for every experiment. Leave it off when a job has to finish by a fixed time, because nobody can promise when spare capacity will be free.

### **Checkpoint 2.1**

Copy `train.py` to `train_deep.py`, change only `max_depth` to 8, and run it. When it finishes, compare the train and validation AUC with the first job. Before you look, write down which of the two numbers you expect to move more. A deeper tree usually raises the training score more than the validation score, and seeing that with your own eyes is the clearest picture of overfitting you will get. Notice that this took one changed line, where a console form would have meant filling in every field again.

**Checkpoint 2.1 — copy the file in the VS Code remote terminal**

In [ ]:
cp train.py train_deep.py

**Checkpoint 2.1 — in train_deep.py, replace the max_depth line with this line**

In [ ]:
        "max_depth": 8,

**Checkpoint 2.1 — run after saving the edited file**

In [ ]:
python train_deep.py

## **3. A real-time endpoint**

Serving a model in SageMaker takes three separate objects, created in order. People new to SageMaker often find this confusing, so here they are in one place.

| Object | What it holds | Bills? |
|---|---|---|
| **Model** | Which container image to run and which `model.tar.gz` to load | No |
| **Endpoint configuration** | Which model, on which machine type, how many machines | No |
| **Endpoint** | The running machines and the HTTPS address | Yes, every hour |

The split exists so that you can change one thing at a time. To move to a bigger machine, you create a new configuration and point the endpoint at it. SageMaker starts the new machines before stopping the old ones, so callers never see a gap.

In `deploy.py` each object is one call. `sagemaker.core.resources` has a Python class for every SageMaker object, and `.create()` on it is the same request a console form sends. The model uses the same container as training, because the XGBoost container both trains and serves.

**[Note]** VS Code, new file `leviathan/deploy.py`. Run it with the model path that `train.py` printed, as in `python deploy.py s3://<your bucket>/output/<job name>/output/model.tar.gz`. It takes about three and a half minutes. While it starts, follow the console steps below the output to show the endpoint being created.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
deploy.py

**File contents — paste into the new file and save**

In [ ]:
"""Serve the trained model on a real-time endpoint: a model, a configuration, then the endpoint."""
import sys
import time

from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core.resources import Endpoint, EndpointConfig, Model
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

from config import REGION

MODEL_DATA = sys.argv[1]  # the s3://.../model.tar.gz that train.py printed

model = Model.create(
    model_name="leviathan-xgb-model",
    primary_container=ContainerDefinition(
        image=image_uris.retrieve("xgboost", REGION, "1.7-1"),
        model_data_url=MODEL_DATA,
    ),
    execution_role_arn=get_execution_role(),
)
print("model:", model.model_name)

config = EndpointConfig.create(
    endpoint_config_name="leviathan-xgb-rt-config",
    production_variants=[ProductionVariant(
        variant_name="AllTraffic",
        model_name=model.model_name,
        instance_type="ml.m5.large",
        initial_instance_count=1,
    )],
)
print("endpoint config:", config.endpoint_config_name)

start = time.time()
endpoint = Endpoint.create(endpoint_name="leviathan-xgb-rt", endpoint_config_name=config.endpoint_config_name)
endpoint.wait_for_status("InService")
print("endpoint:", endpoint.endpoint_name, "InService after", round(time.time() - start), "s")

**Run in the VS Code remote terminal, from the leviathan folder**

Replace the quoted placeholder with the complete model S3 URI printed by train.py.

In [ ]:
python deploy.py "PASTE_MODEL_S3_URI_FROM_TRAIN_PY"

Output from the run, without the SDK's own log lines.

```
model: leviathan-xgb-model
endpoint config: leviathan-xgb-rt-config
endpoint: leviathan-xgb-rt InService after 204 s
```

The endpoint was ready after 204 seconds. A real-time endpoint bills about 0.115 USD every hour from now until it is deleted, about 84 USD a month if it is forgotten.

To watch it in the console while it starts:

1. Type `SageMaker AI` in the search bar at the top of the console and open it.

**AWS console search**

In [ ]:
SageMaker AI

2. In the menu on the left, click **Deployments & inference**. The group starts folded, with only its name showing, and clicking the name opens it. If the whole menu is hidden, click the round menu button at the top left, next to **Amazon SageMaker AI**.
3. Click **Endpoints**. `leviathan-xgb-rt` is in the list with the status **Creating**.

**Endpoint name**

In [ ]:
leviathan-xgb-rt

![nav_menu.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/579/original/nav_menu.png?1791466263)

The same group holds **Deployable models**, where the model from `deploy.py` is listed, and **Endpoint configurations**, where its configuration is. Older guides call this group **Inference** and its first page **Models**. The console no longer uses those names.

4. When the status says **InService**, click `leviathan-xgb-rt` to open its page.

![ep_detail12.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/582/original/ep_detail12.png?1791466271)

The **Endpoint summary** box has the status, the HTTPS address under **URL**, and a link to the container's log under **Model container logs**. You cannot call that URL with a plain `curl`, though. Every request has to be signed with AWS credentials. That is very different from the ECS service, which anybody on the internet could call. Here only someone with IAM permission to invoke this endpoint can use it. If a public website needs these predictions, something in between, usually an API Gateway or your own small service, holds the credentials and calls the endpoint for it. Section 3.4 builds one.

The address is a public AWS name, `runtime.sagemaker.us-west-2.amazonaws.com`. From inside your VPC, private DNS turns that same name into the `sagemaker.runtime` endpoint that section 1 brought back. So the next file calls a public-looking address and its traffic never leaves AWS.

### 3.1 Calling it

You call an endpoint with the `sagemaker-runtime` client. That is a second boto3 client, separate from the `sagemaker` one, which exists only to send requests to endpoints. Its `invoke_endpoint` method takes the endpoint name, the format and the body. The body here is one row of features as a comma-separated line, with no label, in the same order as the training file.

The file sends five held-back orders, then one correct row and three broken ones. The endpoint has no idea what your 18 columns mean. It only counts commas.

**[Note]** VS Code, new file `leviathan/invoke.py`. Run it with `python invoke.py`. Two of the three broken requests are meant to fail.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
invoke.py

**File contents — paste into the new file and save**

In [ ]:
"""Call the real-time endpoint with a few held-back orders, then with three broken rows."""
import time

import boto3
import pandas as pd

from config import BUCKET

s3 = boto3.client("s3")
runtime = boto3.client("sagemaker-runtime")
ENDPOINT = "leviathan-xgb-rt"

rows = pd.read_csv(s3.get_object(Bucket=BUCKET, Key="batch-input/batch.csv")["Body"], header=None)
labels = pd.read_csv(s3.get_object(Bucket=BUCKET, Key="batch-labels/labels.csv")["Body"])["returned"]


def call(body):
    response = runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType="text/csv", Body=body)
    return float(response["Body"].read())


for i in range(5):
    body = ",".join(str(v) for v in rows.iloc[i].values)
    start = time.time()
    p = call(body)
    print(f"p(return)={p:.3f}  actual={labels.iloc[i]}  {round((time.time() - start) * 1000)} ms")

good = rows.iloc[0].tolist()
broken = {
    "one feature too many": good + [1],
    "price left out": good[1:],
    "a raw category value": ["apparel"] + good[1:],
}
print()
print(f"{'correct row':22s} -> p(return)={call(','.join(map(str, good))):.3f}")
for name, values in broken.items():
    try:
        p = call(",".join(map(str, values)))
        print(f"{name:22s} -> ACCEPTED, p(return)={p:.3f}")
    except runtime.exceptions.ModelError as e:
        print(f"{name:22s} -> ModelError: {str(e)[:160]}")

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python invoke.py

Output from the run.

```
p(return)=0.279  actual=1  162 ms
p(return)=0.111  actual=0  380 ms
p(return)=0.127  actual=0  36 ms
p(return)=0.787  actual=0  25 ms
p(return)=0.124  actual=1  15 ms

correct row            -> p(return)=0.279
one feature too many   -> ModelError: An error occurred (ModelError) when calling the InvokeEndpoint operation: Received client error (400) from primary and could not load the entire response body.
price left out         -> ACCEPTED, p(return)=0.069
a raw category value   -> ModelError: An error occurred (ModelError) when calling the InvokeEndpoint operation: Received client error (415) from primary with message "Loading csv data failed with Ex
```

Each answer is the model's probability that the order comes back. The first two calls took 162 and 380 ms and the rest took 15 to 36 ms. These calls came from inside AWS, from the space. The same kind of calls made from a laptop in India, while preparing these classes, took 290 to 590 ms, almost all of it network distance. Where the caller sits matters more than the model here.

The scores themselves are what a 0.7 AUC model looks like. The fourth order got 0.787 and was not returned. The fifth got 0.124 and was returned. The model ranks orders better than chance. It is not a fortune teller.

### 3.2 What the endpoint accepts that it should not

The two loud failures are the safe kind. One column too many gives a 400 error. A word where a number belongs gives a 415 error saying the CSV data could not be loaded. The script cuts each message at 160 characters. The full text is in the endpoint's log in CloudWatch. On the endpoint's page, click the link under **Model container logs** in the **Endpoint summary** box, then the newest log stream. Either one shows up in your logs the first time it happens.

The dangerous one is in the middle. With `price` left out, the row has 17 values instead of 18. The endpoint **accepted it and answered**. Every value moved one column to the left. The discount was read as the price, the delivery days as the discount, and so on. The answer changed from 0.279 to 0.069, a confident "this will not come back" for the wrong reason, and nothing anywhere reported an error.

This happens in real systems when the code that builds requests drifts away from the code that built the training data. Somebody adds a column to one and not the other. The usual fixes are to build both from the same function, or to put a small check in front of the endpoint that counts columns. SageMaker will not do it for you.

### 3.3 The endpoint's own numbers

Every endpoint reports to **CloudWatch**, AWS's service for metrics and logs, once a minute, with no setup. These are the numbers someone on call looks at first. `Invocations` counts requests. `Invocation4XXErrors` counts requests that failed because of the caller, and `Invocation5XXErrors` those that failed because of the endpoint. `ModelLatency` is the time the container spent on a request, and `OverheadLatency` is the time SageMaker itself added around it.

`metrics.py` asks CloudWatch for these over the last 30 minutes, through a third boto3 client called `cloudwatch`. Run it a couple of minutes after `invoke.py`, because the numbers take that long to arrive.

**[Note]** VS Code, new file `leviathan/metrics.py`. Run it with `python metrics.py`. Then show the same numbers in the console. Open **Deployments & inference → Endpoints**, click `leviathan-xgb-rt`, stay on the **Monitor** tab and scroll down to **Invocation metrics**.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
metrics.py

**File contents — paste into the new file and save**

In [ ]:
"""What the endpoint itself reported to CloudWatch since it was created."""
from datetime import datetime, timezone

import boto3

cloudwatch = boto3.client("cloudwatch")
now = datetime.now(timezone.utc)
# Metrics are kept by endpoint name. Start at this endpoint's creation, or an older
# endpoint with the same name gets counted too.
created = boto3.client("sagemaker").describe_endpoint(EndpointName="leviathan-xgb-rt")["CreationTime"]

# name -> (statistic, unit to print). Latencies are reported in microseconds.
METRICS = {
    "Invocations": ("Sum", "requests"),
    "Invocation4XXErrors": ("Sum", "requests"),
    "Invocation5XXErrors": ("Sum", "requests"),
    "ModelLatency": ("Average", "ms"),
    "OverheadLatency": ("Average", "ms"),
}

for name, (stat, unit) in METRICS.items():
    # One point per minute. Sums add up; averages are weighted by how many requests each minute had.
    points = cloudwatch.get_metric_statistics(
        Namespace="AWS/SageMaker",
        MetricName=name,
        Dimensions=[{"Name": "EndpointName", "Value": "leviathan-xgb-rt"},
                    {"Name": "VariantName", "Value": "AllTraffic"}],
        StartTime=created,
        EndTime=now,
        Period=60,
        Statistics=[stat, "SampleCount"],
    )["Datapoints"]
    if not points:
        print(f"{name:20s} no data yet")
        continue
    if stat == "Sum":
        value = sum(p["Sum"] for p in points)
    else:
        value = sum(p["Average"] * p["SampleCount"] for p in points) / sum(p["SampleCount"] for p in points)
    if unit == "ms":
        value = value / 1000
    print(f"{name:20s} {stat.lower():8s} {value:8.1f} {unit}")

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python metrics.py

Output from the run.

```
Invocations          sum           9.0 requests
Invocation4XXErrors  sum           2.0 requests
Invocation5XXErrors  sum           0.0 requests
ModelLatency         average       6.8 ms
OverheadLatency      average      21.3 ms
```

The endpoint saw 9 requests, which is exactly what `invoke.py` sends. Two were counted as client errors, the 400 and the 415. The row with `price` left out is not among them. As far as the endpoint's own numbers go, that request went fine. That is why a check in front of the endpoint matters, because no metric will ever show the problem.

The model itself took 6.8 ms per request on average, and SageMaker added 21.3 ms around it. A caller always measures more than the two together, because the network comes on top.

The first version of this file looked at the last 30 minutes instead of starting at the endpoint's creation time. It reported 10,236 requests. An earlier endpoint with the same name had been load-tested within that half hour, and CloudWatch keeps endpoint metrics by name, not by the endpoint that sent them. So an endpoint that is deleted and created again under the same name inherits the old one's numbers. The console's charts work the same way. To see them:

1. Open **Deployments & inference → Endpoints** and click `leviathan-xgb-rt`.
2. The page opens on the **Monitor** tab. Scroll down past **Operational metrics** to **Invocation metrics**.

![ep_monitor_invocations.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/583/original/ep_monitor_invocations.png?1791466274)

In this picture, taken during the same run, the lines start at 11:00, an hour before this endpoint was created.

### 3.4 A public URL with a check in front

Two problems from this section are still open. The first came up when the endpoint went live. Every call has to be signed with AWS credentials, so a shop's website or mobile app cannot call it directly. The second came up in 3.2 and 3.3. The endpoint scores a row with a column missing, and none of its own numbers notice.

Both have the same fix. A small piece of code sits in front of the endpoint. Callers send it an order as named fields, in JSON. It checks the order, builds the 18 numbers in the order `prepare.py` wrote them, and calls the endpoint itself. Callers never see the CSV row, so they can no longer get the column order wrong. It is the check in front of the endpoint that 3.2 called for, and the list of columns now lives in one place.

![](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/389/original/AWS_Request-to-Endpoint_Flow_Diagram.png?1791891437)

Two AWS services do this. **Lambda** runs a function when a request arrives and bills only for the time it runs, with no server to look after. **API Gateway** gives that function a public HTTPS address. The endpoint does not change. The only new thing allowed to call it is the function's role.

This is not free. Each call now passes through two more services, and you will measure what that costs in time below. A caller that already has AWS credentials, like another service inside your company, can keep calling the endpoint directly. The front door is for callers outside AWS.

**[Note]** Browser. Create the function first.

1. Type `Lambda` in the console search bar and open it. In the left menu click **Create an Event Function** on it.

**AWS console search**

In [ ]:
Lambda

2. Keep **Author from scratch**. Type `leviathan-predict` as the **Function name** and pick **Python 3.14** under **Runtime**. Leave **Permissions** as it is. Lambda makes a new role for the function, named `leviathan-predict-role-` followed by a few random letters, which can only write logs. Click **Create Event Function** at the bottom.

**Function name**

In [ ]:
leviathan-predict

![lambda_create.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/386/original/lambda_create.jpg?1791889956)

3. The function's page opens on the **Code** tab, with a file called `lambda_function.py` in the editor.

**[Note]** Lambda console, the code editor on the **Code** tab, existing file `lambda_function.py`. Select everything in it, paste this cell in its place, and click **Deploy** in the panel on the left of the editor. Nothing runs yet.

**File contents — paste into the Lambda code editor**

In [ ]:
"""Public front for the returns model: check a JSON order, build the row the endpoint expects, return the score."""
import base64
import json
import os

import boto3

# Which endpoint to call. Section 4 changes this setting in the console without touching the code.
ENDPOINT = os.environ.get("ENDPOINT_NAME", "leviathan-xgb-rt")
# The 18 columns prepare.py wrote, in the same order: eight numbers, then the one-hot categories.
NUMBERS = ["price", "discount_pct", "delivery_days", "customer_tenure_days",
           "prior_orders", "prior_returns", "is_gift", "seller_rating"]
CATEGORIES = ["apparel", "beauty", "books", "electronics", "footwear", "home"]
PAYMENTS = ["card", "cod", "upi", "wallet"]

runtime = boto3.client("sagemaker-runtime")


def reply(status, body):
    return {"statusCode": status, "headers": {"Content-Type": "application/json"}, "body": json.dumps(body)}


def lambda_handler(event, context):
    body = event.get("body") or ""
    if event.get("isBase64Encoded"):
        body = base64.b64decode(body).decode()
    try:
        order = json.loads(body)
    except json.JSONDecodeError:
        return reply(400, {"error": "the body must be JSON"})
    if not isinstance(order, dict):
        return reply(400, {"error": "the body must be one JSON object"})

    missing = [name for name in NUMBERS + ["category", "payment_method"] if name not in order]
    if missing:
        return reply(400, {"error": "missing fields", "fields": missing})
    try:
        numbers = [float(order[name]) for name in NUMBERS]
    except (TypeError, ValueError):
        return reply(400, {"error": "these fields must be numbers", "fields": NUMBERS})
    if order["category"] not in CATEGORIES:
        return reply(400, {"error": "unknown category", "allowed": CATEGORIES})
    if order["payment_method"] not in PAYMENTS:
        return reply(400, {"error": "unknown payment_method", "allowed": PAYMENTS})

    row = numbers
    row += [1 if order["category"] == c else 0 for c in CATEGORIES]
    row += [1 if order["payment_method"] == p else 0 for p in PAYMENTS]
    response = runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType="text/csv",
                                       Body=",".join(str(v) for v in row))
    return reply(200, {"p_return": round(float(response["Body"].read()), 3)})

The top of the file is the contract with callers. `ENDPOINT` is the endpoint the function calls. It comes from a setting called `ENDPOINT_NAME`, and while that setting is not there it falls back to `leviathan-xgb-rt`, the endpoint you just built. `NUMBERS`, `CATEGORIES` and `PAYMENTS` are the 18 columns from the previous class, in the same order. `lambda_handler` is the function Lambda calls for every request. It receives the HTTP request as `event`, and whatever it returns becomes the HTTP response. It turns anything wrong into a `400` with a message that names the problem, and only a good order reaches the endpoint. API Gateway marks some request bodies as base64 encoded, depending on how the caller sent them, and the first two lines of the handler undo that.

![lambda_code.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/436/original/lambda_code.jpg?1791897603)

Now the public address.

**[Note]** Browser.

1. Type `API Gateway` in the console search bar and open it. In the left menu click **APIs**, then **Create API**. Under **HTTP API** click **Build**.

**AWS console search**

In [ ]:
API Gateway

2. Type `leviathan-api` as the **API name**. Under **Integrations** click **Add integration** and choose **Lambda**. In **Lambda function** type `leviathan-predict` and pick it from the list. Keep **Version** at **2.0**, which is the request format the handler expects. Click **Next**.

**API name**

In [ ]:
leviathan-api

**Lambda function for the integration**

In [ ]:
leviathan-predict

![api_configure.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/380/original/api_configure.jpg?1791889944)

3. On **Configure routes** click **Add route**. Pick **POST** as the **Method**, type `/predict` as the **Resource path** and pick `leviathan-predict` as the **Integration target**. Click **Next**.

**Resource path**

In [ ]:
/predict

**Integration target**

In [ ]:
leviathan-predict

![api_route.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/382/original/api_route.jpg?1791889948)

4. On **Define stages** keep the stage `$default` with **Auto-deploy** on. A **stage** is one deployed copy of the API, and `$default` serves it at the root of the address. Click **Next**, then **Create**.
5. The API opens on its **Routes** page. In the left menu click **API: leviathan-api** followed by its ID. Under **Stages for leviathan-api**, copy the **Invoke URL**.

![api_invoke_url.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/381/original/api_invoke_url.jpg?1791889946)

Time to call it the way an outside caller would. That means plain `curl`, from a machine with no AWS credentials, over the public internet.

**[Note]** A terminal on your laptop, in any folder. Do not use VS Code's terminal here, because that one runs on the space, inside AWS. Replace the address in the first line with your **Invoke URL**, then paste the whole block. On Windows, use Git Bash.

**Copy into your laptop terminal (Git Bash on Windows)**

In [ ]:
URL=https://nwis6xjyhd.execute-api.us-west-2.amazonaws.com
ORDER='{"price": 14955, "discount_pct": 12, "delivery_days": 5, "customer_tenure_days": 902, "prior_orders": 12, "prior_returns": 2, "is_gift": 0, "seller_rating": 3.4, "category": "electronics", "payment_method": "card"}'
curl -s -w '\nHTTP %{http_code}\n' -X POST $URL/predict -H 'Content-Type: application/json' -d "$ORDER"


Output from the run.

```
{"message":"Internal Server Error"}
HTTP 500
```

The order is the first one in `batch.csv`, the one `invoke.py` scored at 0.279, written out as named fields. The URL works, because API Gateway answered. But the answer is a bare `500`, and that is all a caller ever learns. The reason is in the function's log.

**[Note]** Browser.

1. Go back to the `leviathan-predict` function in Lambda and click the **Monitor** tab, then **View CloudWatch logs**. CloudWatch opens in a new tab, on the function's log group.
2. Under **Log streams**, click the newest stream. Type `ERROR` in the **Filter events** box and press Enter. Click the arrow at the left of a line to open it.

**CloudWatch Filter events**

In [ ]:
ERROR

![lambda_log_denied.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/387/original/lambda_log_denied.jpg?1791889959)

`AccessDeniedException`. The role Lambda made for the function may write logs and nothing else. It is not allowed `sagemaker:InvokeEndpoint`. This is the second time in this class that a role has been the problem, after the training job in section 2. And again nothing on the outside said so. The caller got a generic error, and only the function's own log names the cause.

The fix is a small policy that lets the role call this one endpoint.

**[Note]** Browser.

1. In Lambda, on the function's page, click the **Configuration** tab, then **Permissions** in the list on its left. Under **Execution role**, click the role name. IAM opens in a new tab.
2. Click **Add permissions**, then **Create inline policy**. In the **Policy editor**, switch to **JSON**. Select all the text in it and paste the block below in its place. Click **Next**.
3. Type `leviathan-predict-invoke` as the **Policy name** and click **Create policy**.

**Copy into the IAM JSON policy editor**

In [ ]:
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "sagemaker:InvokeEndpoint",
            "Resource": "arn:aws:sagemaker:us-west-2:*:endpoint/leviathan-xgb-rt"
        }
    ]
}


**Policy name — after clicking Next**

In [ ]:
leviathan-predict-invoke

![iam_inline_json.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/384/original/iam_inline_json.jpg?1791889952)

The policy allows one action on one endpoint. The `*` stands for the account ID, so the same text works in every account. It still names only `leviathan-xgb-rt`.

Now three calls. The good order again, then the order with `price` left out, which is the case that went through silently in 3.2. Last, an order with a category the model has never seen.

**[Note]** The same laptop terminal, so `URL` and `ORDER` are still set. Paste the whole block. IAM changes can take a few seconds to reach Lambda, so if the first call still says `500`, wait ten seconds and paste it again.

**Copy into your laptop terminal (Git Bash on Windows)**

In [ ]:
NO_PRICE='{"discount_pct": 12, "delivery_days": 5, "customer_tenure_days": 902, "prior_orders": 12, "prior_returns": 2, "is_gift": 0, "seller_rating": 3.4, "category": "electronics", "payment_method": "card"}'
TOYS='{"price": 14955, "discount_pct": 12, "delivery_days": 5, "customer_tenure_days": 902, "prior_orders": 12, "prior_returns": 2, "is_gift": 0, "seller_rating": 3.4, "category": "toys", "payment_method": "card"}'
curl -s -w '\nHTTP %{http_code}\n' -X POST $URL/predict -H 'Content-Type: application/json' -d "$ORDER"
curl -s -w '\nHTTP %{http_code}\n' -X POST $URL/predict -H 'Content-Type: application/json' -d "$NO_PRICE"
curl -s -w '\nHTTP %{http_code}\n' -X POST $URL/predict -H 'Content-Type: application/json' -d "$TOYS"


Output from the run.

```
{"p_return": 0.279}
HTTP 200
{"error": "missing fields", "fields": ["price"]}
HTTP 400
{"error": "unknown category", "allowed": ["apparel", "beauty", "books", "electronics", "footwear", "home"]}
HTTP 400
```

The good order scored 0.279, the same as `invoke.py` got from inside the space. It is the same model, now reached from a laptop with no AWS credentials at all. The order with `price` missing is the one 3.2 scored at a confident 0.069. Now it never reaches the model. The caller gets a `400` that names the missing field. The unknown category gets the list of allowed values. Neither of the two bad orders cost an endpoint call.

The extra hop shows up in the timings. While preparing this class, the same calls from a laptop in India took 0.7 to 0.8 seconds each, and the very first one took 1.1 seconds while Lambda started the function. Calling the endpoint directly from the same laptop took 290 to 590 ms, as 3.1 noted. So the front door added roughly 0.2 to 0.4 seconds here. For a person waiting on a web page that is fine. For a system that scores thousands of orders a second, it is a reason to call the endpoint directly from inside AWS.

One more thing before moving on. The URL is now open to anyone who has it, and every request costs a Lambda call and an endpoint call. API Gateway's account limit for this region allows 10,000 requests a second, all of it aimed at one `ml.m5.large`. A **throttle** on the stage caps that.

**[Note]** Browser. In API Gateway, open `leviathan-api`. In its left menu, under **Protect**, click **Throttling**. Pick the stage `$default` and click **Select**. Under **Default route throttling** click **Edit**. Type `10` as the **Burst limit** and `5` as the **Rate limit**, then click **Save**.

**Stage**

In [ ]:
$default

**Burst limit**

In [ ]:
10

**Rate limit**

In [ ]:
5

![api_throttle.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/383/original/api_throttle.jpg?1791889950)

The **rate limit** is the steady number of requests a second the stage accepts. The **burst limit** is how many it lets through at once before the rate applies. Anything above that gets a `429 Too Many Requests` without reaching the function. The page also shows the account's own limits, 10,000 a second, which is what applied before.

**[Note]** The same laptop terminal. Paste the line as it is. It sends 200 requests at once and counts the status codes that come back.

**Copy into your laptop terminal (Git Bash on Windows)**

In [ ]:
seq 200 | xargs -P 200 -I{} curl -s -o /dev/null -w '%{http_code}\n' -X POST $URL/predict -H 'Content-Type: application/json' -d "$ORDER" | sort | uniq -c



Output from the run.

```
  13 200
 187 429
```

13 got an answer and 187 were turned away. The number moves a lot from run to run, though. Across seven floods of 200 sent while preparing this class, anywhere from 13 to all 200 got through. Smaller bursts of 40 at once were hardly ever throttled. API Gateway runs on many servers and applies limits on a best effort basis. Treat throttling as a ceiling against a flood, not as an exact quota.

A throttle protects the endpoint, but the URL is still open to anyone. Real APIs also check who is calling, with API keys, a login token checked by an **authorizer**, or IAM signatures for internal callers. Which one fits depends on who the callers are, and that choice belongs to whoever designs the product. The rest of this section calls the endpoint directly from the space. Callers of the URL need no change through the releases that follow, because the function asks for the endpoint by name.

### 3.5 Releasing a new model without downtime

Models get retrained, and sooner or later the endpoint has to switch to a new `model.tar.gz` while callers keep calling. SageMaker does this with a **blue/green deployment**. The machines serving now are called blue. SageMaker starts a second set, called green, with the new model, moves the traffic to it, and keeps blue running until it is sure. If something goes wrong, the traffic goes back to blue.

`update.py` releases the model from the spot job in 2.2. It creates a new model and a new endpoint configuration, the same way `deploy.py` did, then points the endpoint at the new configuration with three settings.

- `ALL_AT_ONCE` moves all the traffic to green in one step.
- `wait_interval_in_seconds=120` asks for a **baking period** of two minutes. During it green takes all the traffic while blue keeps running, ready to take it back.
- `auto_rollback_configuration` names a CloudWatch alarm. If the alarm goes off during the baking period, SageMaker moves the traffic back to blue by itself. The file creates that alarm first. It goes off when the endpoint returns any 5XX error within a minute.

While the update runs, a thread in the file keeps calling the endpoint twice a second, the way real users would, and counts the answers and the failures.

The first version of this file used `CANARY`, which sends only part of the traffic to green first and moves the rest only if that part does well. AWS refused it.

```
ClientError: An error occurred (ValidationException) when calling the
UpdateEndpoint operation: Cannot create/update endpoint with single instance of
variant AllTraffic, when using CANARY traffic shifting type in the specified
DeploymentConfig. Please increase the instance count and retry the request, or
switch to ALL-AT-ONCE traffic shifting.
```

A canary needs at least two machines, so that one of them can go first. With one machine, all at once plus a baking period and an alarm is the safe choice.

**[Note]** VS Code, new file `leviathan/update.py`. Run it with the model path that `train_spot.py` printed, as in `python update.py s3://<your bucket>/output/<spot job name>/output/model.tar.gz`. It takes about five minutes. While it runs, open the endpoint's page from **Deployments & inference → Endpoints**. Show **Status** saying **Updating**, then click the **Configuration** tab, where **Endpoint configuration** still names the old configuration until the switch.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
update.py

**File contents — paste into the new file and save**

In [ ]:
"""Move the real-time endpoint to a new model blue/green, and roll back by itself on errors."""
import sys
import threading
import time

import boto3
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core.resources import Endpoint, EndpointConfig, Model
from sagemaker.core.shapes import (Alarm, AutoRollbackConfig, BlueGreenUpdatePolicy,
                                   ContainerDefinition, DeploymentConfig, ProductionVariant,
                                   TrafficRoutingConfig)

from config import REGION

MODEL_DATA = sys.argv[1]  # the new model.tar.gz, from train_spot.py
ENDPOINT = "leviathan-xgb-rt"
ALARM = "leviathan-xgb-rt-5xx"

model = Model.create(
    model_name="leviathan-xgb-model-v2",
    primary_container=ContainerDefinition(image=image_uris.retrieve("xgboost", REGION, "1.7-1"),
                                          model_data_url=MODEL_DATA),
    execution_role_arn=get_execution_role(),
)
config = EndpointConfig.create(
    endpoint_config_name="leviathan-xgb-rt-config-v2",
    production_variants=[ProductionVariant(variant_name="AllTraffic", model_name=model.model_name,
                                           instance_type="ml.m5.large", initial_instance_count=1)],
)

# The alarm that decides a rollback: any server error from the endpoint in a minute.
boto3.client("cloudwatch").put_metric_alarm(
    AlarmName=ALARM, Namespace="AWS/SageMaker", MetricName="Invocation5XXErrors",
    Dimensions=[{"Name": "EndpointName", "Value": ENDPOINT}, {"Name": "VariantName", "Value": "AllTraffic"}],
    Statistic="Sum", Period=60, EvaluationPeriods=1, Threshold=0,
    ComparisonOperator="GreaterThanThreshold", TreatMissingData="notBreaching",
)

# Keep calling the endpoint the whole time, the way real users would.
runtime = boto3.client("sagemaker-runtime")
row = "14955.0,12,5,902,12,2,0,3.4,0,0,0,1,0,0,1,0,0,0"  # the first order in batch.csv
calls = {"ok": 0, "failed": 0}
done = threading.Event()

def traffic():
    while not done.is_set():
        try:
            runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType="text/csv", Body=row)["Body"].read()
            calls["ok"] += 1
        except Exception:
            calls["failed"] += 1
        time.sleep(0.5)

threading.Thread(target=traffic, daemon=True).start()

endpoint = Endpoint.get(ENDPOINT)

print("serving before:", endpoint.endpoint_config_name)

endpoint.endpoint_config_name = config.endpoint_config_name

endpoint.update(
    deployment_config=DeploymentConfig(
        blue_green_update_policy=BlueGreenUpdatePolicy(
            traffic_routing_configuration=TrafficRoutingConfig(
                type="ALL_AT_ONCE",
                wait_interval_in_seconds=0,
            ),
            termination_wait_in_seconds=180,
        ),
        auto_rollback_configuration=AutoRollbackConfig(
            alarms=[Alarm(alarm_name=ALARM)]
        ),
    )
)

sm = boto3.client("sagemaker")
start, last = time.time(), None
while True:
    d = sm.describe_endpoint(EndpointName=ENDPOINT)
    pending = d.get("PendingDeploymentSummary", {}).get("ProductionVariants", [])
    stage = pending[0].get("VariantStatus", [{}])[0].get("Status", "-") if pending else "-"
    now = (d["EndpointStatus"], stage)
    if now != last:
        print(f"after {round(time.time() - start)} s: endpoint {now[0]}, new fleet {now[1]}")
        last = now
    if d["EndpointStatus"] in {
    "InService",
    "Failed",
    "UpdateRollbackFailed"
    }:
        break
    
    time.sleep(10)
done.set()

print("serving after:", d["EndpointConfigName"])
print(f"calls during the update: {calls['ok']} answered, {calls['failed']} failed")

**Run in the VS Code remote terminal, from the leviathan folder**

Replace the quoted placeholder with the complete model S3 URI printed by train_spot.py.

In [ ]:
python update.py "PASTE_MODEL_S3_URI_FROM_TRAIN_SPOT_PY"

Output from the run, without the SDK's own log lines.

```
serving before: leviathan-xgb-rt-config
after 1 s: endpoint Updating, new fleet -
after 12 s: endpoint Updating, new fleet Creating
after 260 s: endpoint Updating, new fleet ActivatingTraffic
after 271 s: endpoint Updating, new fleet Baking
after 336 s: endpoint InService, new fleet -
serving after: leviathan-xgb-rt-config-v2
calls during the update: 351 answered, 0 failed
```

The whole update took 336 seconds. For the first 260, SageMaker was creating the green fleet, which is the same work as a fresh deploy. Then it moved the traffic across and green went into its baking period, with blue still standing by. The alarm stayed quiet, so at 336 seconds the endpoint was back to InService and serving `leviathan-xgb-rt-config-v2`. The file checks the status every ten seconds, so these times are rough. In a second run, from the space, the two short stages printed in the other order, Baking before ActivatingTraffic, and the update took 285 seconds with 555 calls answered and none failed.

The thread called the endpoint 351 times during the update, and not one call failed. That is what blue/green is for. The callers never noticed the switch. The new model has the same AUC as the old one, because 2.2 trained it with the same settings. In real work the new model differs, and the baking period and the alarm are there for the case where it is worse.

To watch an update in the console:

1. Open **Deployments & inference → Endpoints** and click `leviathan-xgb-rt`. **Status** in **Endpoint summary** says **Updating**.
2. Click the **Configuration** tab. **Endpoint configuration** still names `leviathan-xgb-rt-config`, and **Endpoint runtime settings** still shows the old model, `leviathan-xgb-model`, until the switch.

![ep_updating_cfg.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/584/original/ep_updating_cfg.png?1791466277)

Not every release works, and a failed one is harder to spot than you would expect. While preparing this class, a release pointed the endpoint at a `model.tar.gz` with the right file name inside and the wrong contents, the way a cut-off upload looks. SageMaker only checks that the file exists, so the update got as far as starting new machines. Each one failed to load the model, and SageMaker kept trying. After 1,420 seconds, nearly 24 minutes of billed machines, it gave up.

```
after 1420 s the endpoint is InService
serving: leviathan-xgb-rt-config-v2
failure reason: The primary container for production variant AllTraffic did not pass the ping health check. Please check CloudWatch logs for this endpoint.
```

Callers never noticed, because blue kept serving the whole time. But look at the status. InService looks like success. A deploy script that waits for InService and stops there would report this release as done. After every release, compare `EndpointConfigName` with the configuration you meant to release, and read `FailureReason`. The cause was in the container's log, under **Model container logs** on the endpoint's page.

```
RuntimeError: Model /opt/ml/model/xgboost-model cannot be loaded:
Pickle load error=could not find MARK
```

Catching that before a release is cheap. Load the model file once in a test, the way the container would, before pointing an endpoint at it. In the console, a failed release adds a fourth tab, **Errors**, to the endpoint's page, next to **Monitor**, **Configuration** and **Alarms**. It shows the same failure reason and when it happened.

![ep_failed_update.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/586/original/ep_failed_update.png?1791466283)

### 3.6 Autoscaling

A real-time endpoint has as many machines as its configuration says, one here, however busy it gets. **Autoscaling** changes that number with the traffic. SageMaker does not do this itself. A separate AWS service, **Application Auto Scaling**, does it for SageMaker, ECS and a few others, which is why `autoscale.py` uses a fourth boto3 client, `application-autoscaling`.

The file makes two calls. `register_scalable_target` says what may change, which is the variant's machine count, and between which limits, here 1 and 2. `put_scaling_policy` says how. A **target tracking** policy names a metric and a value to hold it near, here 300 requests a minute for each machine. AWS then creates two CloudWatch alarms by itself, one that adds a machine and one that removes one.

`load.py` then sends steady traffic from eight threads for up to 15 minutes. Once a minute it prints how many requests went out since the last line, and how many machines the endpoint wants and has.

**[Note]** VS Code, new file `leviathan/autoscale.py`. Run it with `python autoscale.py`.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
autoscale.py

**File contents — paste into the new file and save**

In [ ]:
"""Let the real-time endpoint add a second machine when traffic per machine passes a target."""
import boto3

ENDPOINT = "leviathan-xgb-rt"
RESOURCE = f"endpoint/{ENDPOINT}/variant/AllTraffic"

scaling = boto3.client("application-autoscaling")
œ
# The range the endpoint may move in. It never goes below 1 or above 2 machines.
scaling.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=RESOURCE,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=1,
    MaxCapacity=2,
)

# Target tracking: keep each machine near 300 requests a minute, adding or removing machines to stay there.
scaling.put_scaling_policy(
    PolicyName="leviathan-xgb-rt-invocations",
    ServiceNamespace="sagemaker",
    ResourceId=RESOURCE,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 300.0, # 300 requests per minute per machine
        "PredefinedMetricSpecification": {"PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"},
        "ScaleOutCooldown": 60,
        "ScaleInCooldown": 300,
    },
)

for target in scaling.describe_scalable_targets(ServiceNamespace="sagemaker", ResourceIds=[RESOURCE])["ScalableTargets"]:
    print("scalable target:", target["ResourceId"], "min", target["MinCapacity"], "max", target["MaxCapacity"])
alarms = scaling.describe_scaling_policies(ServiceNamespace="sagemaker", ResourceId=RESOURCE)["ScalingPolicies"][0]["Alarms"]
for alarm in alarms:
    print("alarm AWS created for the policy:", alarm["AlarmName"])

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python autoscale.py

Output from the run.

```
scalable target: endpoint/leviathan-xgb-rt/variant/AllTraffic min 1 max 2
alarm AWS created for the policy: TargetTracking-endpoint/leviathan-xgb-rt/variant/AllTraffic-AlarmHigh-cacd135b-3b9d-4f49-9a57-d28f34114327
alarm AWS created for the policy: TargetTracking-endpoint/leviathan-xgb-rt/variant/AllTraffic-AlarmLow-a7e0741f-c2eb-4464-965b-01821676f0f0
```

The two alarm names end in `AlarmHigh` and `AlarmLow`. They show up in CloudWatch under **Alarms**, and AWS deletes them by itself when the scaling policy goes.

**[Note]** VS Code, new file `leviathan/load.py`. Run it with `python load.py`. It runs for up to fifteen minutes and stops early once a second machine is running. Meanwhile open the endpoint's page, click the **Configuration** tab and show **Desired instance count** and **Current instance count** under **Endpoint runtime settings**. On a narrow screen those columns sit off to the right, so scroll the table sideways. Refresh the page every few minutes.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
load.py

**File contents — paste into the new file and save**

In [ ]:
"""Send steady traffic for up to 15 minutes and watch the endpoint add a machine."""
import threading
import time

import boto3

ENDPOINT = "leviathan-xgb-rt"
row = "14955.0,12,5,902,12,2,0,3.4,0,0,0,1,0,0,1,0,0,0"  # the first order in batch.csv
runtime = boto3.client("sagemaker-runtime")
sm = boto3.client("sagemaker")
sent = [0]
done = threading.Event()

def worker():
    while not done.is_set():
        runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType="text/csv", Body=row)["Body"].read()
        sent[0] += 1
        time.sleep(0.05)

for _ in range(8):
    threading.Thread(target=worker, daemon=True).start()

start = last = time.time()
before = 0
while time.time() - start < 15 * 60:
    time.sleep(60)
    variant = sm.describe_endpoint(EndpointName=ENDPOINT)["ProductionVariants"][0]
    now = time.time()
    rate = (sent[0] - before) / (now - last) * 60  # requests a minute since the last line
    before, last = sent[0], now
    print(f"minute {round((now - start) / 60)}: {rate:.0f} requests a minute, "
          f"machines wanted {variant['DesiredInstanceCount']}, running {variant['CurrentInstanceCount']}")
    if variant["CurrentInstanceCount"] == 2:
        break
done.set()
print("total requests:", sent[0])

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python load.py

Output from the run.

```
minute 1: 7536 requests a minute, machines wanted 1, running 1
minute 2: 7922 requests a minute, machines wanted 1, running 1
minute 3: 7971 requests a minute, machines wanted 1, running 1
minute 4: 7948 requests a minute, machines wanted 1, running 1
minute 5: 7954 requests a minute, machines wanted 2, running 1
minute 6: 7928 requests a minute, machines wanted 2, running 1
minute 7: 7952 requests a minute, machines wanted 2, running 1
minute 8: 7792 requests a minute, machines wanted 2, running 2
total requests: 63174
```

The traffic stayed near 8,000 requests a minute, more than 25 times the target of 300 for one machine. Even so, nothing happened for four minutes. The alarm that adds a machine needs three one-minute periods in a row above the target, and the metric itself reaches CloudWatch a minute or two late. At minute 5 the endpoint wanted a second machine, and at minute 8 it had one, because starting a machine and loading the model takes a few minutes too.

So autoscaling reacts in minutes, not seconds. It copes with traffic that grows through the day. It will not save you from a sudden spike, which is why teams keep the minimum at two or more machines for anything that matters. Going back down is slower on purpose. The other alarm waits for 15 minutes in a row below 270 requests a minute per machine before it removes one, so a short quiet spell does not shrink the endpoint just before traffic returns.

These requests came from the space, which sits next to the endpoint. An earlier run of the same file from a laptop in India managed only about 1,600 requests a minute, because each call spent most of its time crossing the network. Even that was five times the target, and the endpoint wanted a second machine at minute 6 and had it at minute 9. How fast the traffic arrives changes the numbers a little. The three-minute wait and the slow start of the new machine do not change.

To see the second machine in the console:

1. Open **Deployments & inference → Endpoints**, click `leviathan-xgb-rt` and click the **Configuration** tab.
2. Scroll down to **Endpoint runtime settings**. **Current instance count** and **Desired instance count** both say 2, and the last column shows the range from `autoscale.py`, 1 to 2. On a narrow screen these columns sit off to the right, so scroll the table sideways.

![ep_scaled_cfg.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/585/original/ep_scaled_cfg.png?1791466280)

## **4. A serverless endpoint**

A real-time endpoint keeps a machine running all day. If your model gets a few hundred requests a day, you are paying for 24 hours of machine to do a few minutes of work. A **serverless endpoint** has no machine of its own. SageMaker starts one when a request arrives and stops it after a while with no traffic. You pay only for the time spent handling requests.

The setup is the same model with a different endpoint configuration. The variant has a memory size and a limit on how many requests it handles at once, instead of a machine type.

The first version of this file built that configuration with `EndpointConfig.create`, the same way as `deploy.py`. It failed before reaching AWS.

```
ParamValidationError: Parameter validation failed:
Missing required parameter in ProductionVariants[0].ServerlessConfig: "MemorySizeInMB"
Unknown parameter in ProductionVariants[0].ServerlessConfig: "MemorySizeInMb", must be one of: MemorySizeInMB, MaxConcurrency, ProvisionedConcurrency
```

That is a bug in the installed SDK, `sagemaker-core` 2.21.0. It spells one field name with a small `b`, and the API wants `MB`. Nothing about your code is wrong. The fix teams use is to drop down to **boto3** for that one call. The SageMaker SDK is built on boto3, and boto3 takes the API's field names exactly as AWS documents them. So `serverless.py` makes the configuration with boto3 and the endpoint with the SDK.

**[Note]** VS Code, new file `leviathan/serverless.py`. Run it with `python serverless.py`. It takes about three minutes.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
serverless.py

**File contents — paste into the new file and save**

In [ ]:
"""Serve the same model on a serverless endpoint."""
import time

import boto3
from sagemaker.core.resources import Endpoint

# sagemaker-core 2.21.0 sends the memory field as "MemorySizeInMb" and the API rejects it,
# so this one call uses boto3, the plain AWS library the SageMaker SDK is built on.
boto3.client("sagemaker").create_endpoint_config(
    EndpointConfigName="leviathan-xgb-sl-config",
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": "leviathan-xgb-model",
        "ServerlessConfig": {"MemorySizeInMB": 1024, "MaxConcurrency": 20},
    }],
)

start = time.time()
endpoint = Endpoint.create(endpoint_name="leviathan-xgb-sl", endpoint_config_name="leviathan-xgb-sl-config")
endpoint.wait_for_status("InService")
print("endpoint:", endpoint.endpoint_name, "InService after", round(time.time() - start), "s")

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python serverless.py

Output from the run, without the SDK's own log lines.

```
endpoint: leviathan-xgb-sl InService after 166 s
```

The catch is the first request after a quiet spell. The model has to be loaded onto a fresh machine before it can answer. This is called a **cold start**. Straight after it is created, a serverless endpoint answers warm, which makes a test run at that moment look fast. Measure it after it has sat idle, the way real traffic will meet it. Section 5.1 does that, after section 5 has given it time to go quiet.

### 4.1 Moving the public URL onto it

The URL from 3.4 still sends every order to `leviathan-xgb-rt`. Callers do not know which endpoint answers them, and they should not have to. So you can move them to the serverless endpoint without telling them, and the URL stays exactly the same. Three settings change, all on the function's side.

The first is the endpoint's name. The function reads it from the `ENDPOINT_NAME` setting, which has not existed so far, so it fell back to `leviathan-xgb-rt`. The second is the role. Its policy names `leviathan-xgb-rt` only, so it has to allow the new endpoint too.

The third is easy to miss. Lambda stops a function after 3 seconds unless told otherwise. The real-time endpoint answers in milliseconds, so 3 seconds never mattered. A serverless endpoint that has gone cold takes longer than that. While preparing this class, the URL was first moved with the timeout left at 3 seconds. Fifteen quiet minutes later, two orders sent one after the other both came back like this.

```
{"message":"Internal Server Error"}
HTTP 500  4.445380 s
{"message":"Internal Server Error"}
HTTP 500  4.072023 s
```

The function's log had one line for each call that ended like this.

```
REPORT RequestId: c171f11f-c8c6-46b5-adff-d77b3e0693da	Duration: 3000.00 ms	Billed Duration: 3502 ms	Memory Size: 128 MB	Max Memory Used: 94 MB	Init Duration: 501.67 ms	Status: timeout
```

`Duration: 3000.00 ms` and `Status: timeout`. Lambda gave up on the function at exactly 3 seconds, while the endpoint was still waking up, and the caller got the same bare `500` as the missing permission did in 3.4. Even the second caller, a second later, failed the same way. So the timeout goes up as well. 30 seconds is the longest API Gateway itself waits for a function, so there is no point setting it higher.

**[Note]** Browser. All three changes are on the `leviathan-predict` function in Lambda, on its **Configuration** tab.

1. Click **Environment variables** in the list on the left, then **Edit**. Click **Add environment variable**. Type `ENDPOINT_NAME` as the **Key** and `leviathan-xgb-sl` as the **Value**, then click **Save**.

**Environment variable — Key**

In [ ]:
ENDPOINT_NAME

**Environment variable — Value**

In [ ]:
leviathan-xgb-sl

![lambda_env_var.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/434/original/lambda_env_var.jpg?1791897538)

2. Click **General configuration**, then **Edit**. Under **Timeout**, set it to `0` min and `30` sec, then click **Save**.

**Timeout — minutes**

In [ ]:
0

**Timeout — seconds**

In [ ]:
30

![lambda_timeout.jpg](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/435/original/lambda_timeout.jpg?1791897541)

3. Click **Permissions**, then the role name under **Execution role**. In IAM, under **Permissions policies**, click `leviathan-predict-invoke`. It opens in a new tab, on **Modify permissions in leviathan-predict-invoke**. Select all the text in the JSON editor, paste the block below in its place, click **Next**, then **Save changes**.

**Policy to open**

In [ ]:
leviathan-predict-invoke

**Copy into the IAM JSON policy editor**

In [ ]:
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "sagemaker:InvokeEndpoint",
            "Resource": [
                "arn:aws:sagemaker:us-west-2:*:endpoint/leviathan-xgb-rt",
                "arn:aws:sagemaker:us-west-2:*:endpoint/leviathan-xgb-sl"
            ]
        }
    ]
}


Now the same order through the same URL. The only difference from 3.4 is that `curl` also prints how long the whole call took.

**[Note]** The laptop terminal from 3.4, where `URL` and `ORDER` are still set. If you closed it, paste the two lines that set them in 3.4 again first. Then paste this line as it is.

**Copy into your laptop terminal (Git Bash on Windows)**

In [ ]:
curl -s -w '\nHTTP %{http_code}  %{time_total} s\n' -X POST $URL/predict -H 'Content-Type: application/json' -d "$ORDER"


Output from the run, right after the serverless endpoint was created.

```
{"p_return": 0.279}
HTTP 200  2.270505 s
```

The same 0.279 from the same URL, now answered by the serverless endpoint. Nobody calling the URL had to change anything. The 2.3 seconds is slower than the 0.7 to 0.8 seconds of 3.4, because the function itself was starting up too. Its log reported `Init Duration: 1019.66 ms` for that call. A serverless endpoint that was just created has not gone cold yet. Section 5.1 measures what happens after it has.

## **5. Batch transform**

Often nobody needs an answer this second. The shop wants a return-risk score for every order placed yesterday, to plan warehouse staff. For that you do not need an endpoint at all, and no public URL either. Sending 1,800 orders through the URL one by one would mean 1,800 function calls for an answer nobody is waiting for. A **batch transform job** starts a machine, loads the model, scores every row in an S3 prefix, writes the scores to another prefix and stops.

Two settings in `batch.py` are worth reading. `split_type="Line"` tells SageMaker to send the file to the model a few lines at a time, instead of as one huge request. `assemble_with="Line"` puts the answers back in the same order, one per line. Without it the output is all the scores run together. After the job, the file reads the scores and the labels that `prepare.py` kept aside and checks the AUC.

**[Note]** VS Code, new file `leviathan/batch.py`. Run it with `python batch.py`. It takes about four and a half minutes. While it runs, switch to the console. In SageMaker AI's left menu, click **Deployments & inference** to open the group if it is folded, then click **Batch transform jobs**. The new job is at the top, named `leviathan-batch-` plus a number. Click it to open its page. **Job summary** at the top shows its status, and the **Monitor** tab has **View logs**, which opens the job's log in CloudWatch. Then start on Checkpoint 5.1 with the class.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
batch.py

**File contents — paste into the new file and save**

In [ ]:
"""Score the held-back file with a batch transform job, then check the scores against the labels."""
import time

import boto3
import pandas as pd
from sagemaker.core.resources import TransformJob
from sagemaker.core.shapes import (TransformDataSource, TransformInput, TransformOutput,
                                   TransformResources, TransformS3DataSource)
from sklearn.metrics import roc_auc_score

from config import BUCKET

name = "leviathan-batch-" + time.strftime("%H%M%S")
start = time.time()
job = TransformJob.create(
    transform_job_name=name,
    model_name="leviathan-xgb-model",
    transform_input=TransformInput(
        data_source=TransformDataSource(s3_data_source=TransformS3DataSource(
            s3_data_type="S3Prefix", s3_uri=f"s3://{BUCKET}/batch-input/")),
        content_type="text/csv",
        split_type="Line",
    ),
    transform_output=TransformOutput(s3_output_path=f"s3://{BUCKET}/batch-output/", assemble_with="Line"),
    transform_resources=TransformResources(instance_type="ml.m5.large", instance_count=1),
)
job.wait()
print("job:", name, job.transform_job_status, "after", round(time.time() - start), "s")

s3 = boto3.client("s3")
out = s3.get_object(Bucket=BUCKET, Key="batch-output/batch.csv.out")["Body"].read().decode()
scores = [float(x) for x in out.split()]
labels = pd.read_csv(s3.get_object(Bucket=BUCKET, Key="batch-labels/labels.csv")["Body"])["returned"]

print("rows scored:", len(scores), " labels:", len(labels))
print("first five scores:", [round(s, 3) for s in scores[:5]])
print("AUC on the batch rows:", round(roc_auc_score(labels, scores), 3))

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python batch.py

Output from the run, without the SDK's own log lines.

```
job: leviathan-batch-072235 Completed after 262 s
rows scored: 1800  labels: 1800
first five scores: [0.279, 0.111, 0.127, 0.787, 0.124]
AUC on the batch rows: 0.726
```

All 1,800 rows came back with a score, in the same order as they went in. The first five match the real-time endpoint's answers exactly. The AUC on these rows, which the model never saw during training, is 0.726. That is close to the 0.708 validation score, so the model is behaving on new data the way training said it would.

To find the job in the console:

1. Type `SageMaker AI` in the search bar and open it.

**AWS console search**

In [ ]:
SageMaker AI

2. In the menu on the left, click **Deployments & inference** to open the group, then click **Batch transform jobs**. The newest job is at the top, named `leviathan-batch-` plus a number.
3. Click its name. **Job summary** shows the status and roughly how long it ran, and **Job configuration** below shows the model it used, `leviathan-xgb-model`, and the machine. The **Monitor** tab has **View logs**, which opens the job's log in CloudWatch.

![bt_detail12.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/587/original/bt_detail12.png?1791466285)

 The picture is from a later run of the same file, which is why its name differs from the output above.

The job took 262 seconds. Most of that went into starting the machine, and the scoring itself took 109 seconds. The output file is named after the input file with `.out` added, here `batch.csv.out`. If the input prefix holds several files, you get one `.out` file for each.

### 5.1 The cold start

The serverless endpoint has now been left alone for a while. Time to find out what its first caller waits for. That caller is anyone on the public URL, so measure it there, the way a customer would meet it.

**[Note]** The laptop terminal from 3.4 and 4.1. Paste the line as it is once the serverless endpoint has been idle for at least ten minutes. The batch job and the checkpoint below usually fill that time. If they did not, finish the checkpoint discussion first. It sends the same order four times in a row and prints how long each call took.

**Copy into your laptop terminal (Git Bash on Windows)**

In [ ]:
for i in 1 2 3 4; do curl -s -w '  %{time_total} s\n' -X POST $URL/predict -H 'Content-Type: application/json' -d "$ORDER"; done


Output from the run, after 15 idle minutes.

```
{"p_return": 0.279}  12.275158 s
{"p_return": 0.279}  0.775827 s
{"p_return": 0.279}  0.742726 s
{"p_return": 0.279}  0.715132 s
```

The first caller waited 12.3 seconds. The next three got their answers in 0.7 to 0.8 seconds, the same as the real-time endpoint gave through the URL in 3.4. All four got 0.279, the same as every other way of serving in this class, because it is the same model file. Only the hosting is different.

Most of those 12 seconds is SageMaker loading the model onto a fresh machine. While preparing this class, a script that called the serverless endpoint directly from the space saw 11 seconds for its first call and about 40 ms for the next ones. The rest of the 12.3 seconds is the trip from the laptop and the function. And this is exactly the call that Lambda's default 3-second timeout would have turned into a `500`, which is why 4.1 raised it.

| Option | You pay for | Use it when |
|---|---|---|
| Real-time endpoint | Every hour the machine exists | Steady traffic, or every request must be fast |
| Serverless endpoint | Only the time spent on requests | Few or bursty requests, and a slow first answer is acceptable |
| Batch transform | Only while the job runs | You have a whole file to score and nobody is waiting |

Neither endpoint is the right answer in general. A fraud check at checkout cannot wait 12 seconds, so it needs a real-time endpoint. A return-risk score shown to the warehouse team in a dashboard can live with a slow first load. Decide from the traffic and the cost of waiting, not from which one is newer.

### **Checkpoint 5.1**

You have now seen three ways to serve the same model. For each of these, decide which one you would use and say why in one line. First, a "chance of return" badge on every product page. Second, a nightly email to the warehouse with tomorrow's likely returns. Third, an internal tool that a support agent opens maybe twenty times a day. There is no single right answer for the third. Being able to argue it either way is the point.

## **6. Tearing it all down**

Start with the public pieces from 3.4. The URL is open to anyone, and once the endpoint behind it is gone it could only answer with errors. Deleting the function does not delete everything Lambda made for it. Its own delete dialog says that the function's logs and role are kept. So there are five things to remove, in this order.

**[Note]** Browser.

1. In API Gateway, click **APIs** in the left menu. Select `leviathan-api`, click **Delete**, type `confirm` and click **Delete**.

**API to delete**

In [ ]:
leviathan-api

**API deletion confirmation**

In [ ]:
confirm

2. In Lambda, open the list of functions. Select `leviathan-predict`, choose **Actions → Delete**, type `confirm` and click **Delete**.

**Function to delete**

In [ ]:
leviathan-predict

**Function deletion confirmation**

In [ ]:
confirm

3. In IAM, click **Roles** in the left menu and type `leviathan-predict-role` in the search box. Select the role, click **Delete**, type the role's full name and click **Delete**. Its inline policy goes with it.

**IAM role search prefix — use the complete role name shown in your console for deletion**

In [ ]:
leviathan-predict-role

4. Lambda also made a separate policy for the role, named `AWSLambdaBasicExecutionRole-` followed by a long ID, and it stays behind when the role goes. In IAM, click **Policies** in the left menu, type `AWSLambdaBasicExecutionRole` in the search box, select it and delete it.

**IAM policy search**

In [ ]:
AWSLambdaBasicExecutionRole

5. In CloudWatch, under **Logs** in the left menu, click **Log Management**. Select `/aws/lambda/leviathan-predict`, choose **Actions → Delete log group(s)** and click **Delete**.

**CloudWatch log group**

In [ ]:
/aws/lambda/leviathan-predict

Four kinds of thing left bill by the hour. Those are the two SageMaker endpoints, the space, the NAT gateway and the interface endpoints. Delete the SageMaker endpoints first, because they cost the most and nothing else depends on them. That is also code.

`cleanup.py` removes things in a set order. Autoscaling goes first. Taking away the scalable target also deletes the two alarms AWS made for the policy, and it stops anything from trying to resize an endpoint that is about to disappear. Then it deletes the rollback alarm from 3.5, the two endpoints, the three endpoint configurations and the two models. Models and configurations cost nothing, but leaving them makes the next person's console harder to read.

**[Note]** VS Code, new file `leviathan/cleanup.py`. Run it with `python cleanup.py`. Run it only after section 5.1, so the serverless endpoint has been measured.

**New file name — create this file in the leviathan folder in VS Code**

In [ ]:
cleanup.py

**File contents — paste into the new file and save**

In [ ]:
"""Delete everything that serving created. The endpoints are what bill by the hour."""
import boto3
from sagemaker.core.resources import Endpoint, EndpointConfig, Model

# Scaling first: a scalable target left behind keeps pointing at an endpoint that no longer exists.
scaling = boto3.client("application-autoscaling")
try:
    scaling.deregister_scalable_target(ServiceNamespace="sagemaker",
                                       ResourceId="endpoint/leviathan-xgb-rt/variant/AllTraffic",
                                       ScalableDimension="sagemaker:variant:DesiredInstanceCount")
    print("removed autoscaling from leviathan-xgb-rt")
except scaling.exceptions.ObjectNotFoundException:
    print("no autoscaling on leviathan-xgb-rt")

boto3.client("cloudwatch").delete_alarms(AlarmNames=["leviathan-xgb-rt-5xx"])
print("deleted alarm leviathan-xgb-rt-5xx")

for name in ["leviathan-xgb-rt", "leviathan-xgb-sl"]:
    try:
        Endpoint.get(name).delete()
        print("deleted endpoint", name)
    except Exception as e:
        print("endpoint", name, "not found:", type(e).__name__)

for name in ["leviathan-xgb-rt-config", "leviathan-xgb-rt-config-v2", "leviathan-xgb-sl-config"]:
    try:
        EndpointConfig.get(name).delete()
        print("deleted endpoint config", name)
    except Exception as e:
        print("endpoint config", name, "not found:", type(e).__name__)

for name in ["leviathan-xgb-model", "leviathan-xgb-model-v2"]:
    try:
        Model.get(name).delete()
        print("deleted model", name)
    except Exception as e:
        print("model", name, "not found:", type(e).__name__)

**Run in the VS Code remote terminal, from the leviathan folder**

In [ ]:
python cleanup.py

Output from the run, without the SDK's own log lines.

```
removed autoscaling from leviathan-xgb-rt
deleted alarm leviathan-xgb-rt-5xx
deleted endpoint leviathan-xgb-rt
deleted endpoint leviathan-xgb-sl
deleted endpoint config leviathan-xgb-rt-config
deleted endpoint config leviathan-xgb-rt-config-v2
deleted endpoint config leviathan-xgb-sl-config
deleted model leviathan-xgb-model
deleted model leviathan-xgb-model-v2
```

Check it with your own eyes, because a script that failed half way would have printed an error you might have missed.

1. Open **Deployments & inference → Endpoints**. The list is empty.

![ep_list_empty.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/975/original/ep_list_empty.png?1791359494)

2. Click **Endpoint configurations**, then **Deployable models**, in the same group. Both should be empty too.

**[Note]** Laptop and browser. Close the remote VS Code window. Then in the Studio tab open **JupyterLab**, open `leviathan-lab` and click **Stop space**, then **Stop space** in the dialog.

**Space to stop**

In [ ]:
leviathan-lab

![space_stop_dialog.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/991/original/space_stop_dialog.png?1791359532)

The blue box in that dialog is the most important sentence in these two classes. Stopping the space does **not** delete endpoints, buckets or anything else you created from it. It only stops the space's own machine. Each thing you created has to be removed where it lives.

![space_stopping.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/992/original/space_stopping.png?1791359534)

**[Note]** Browser. Delete the interface endpoints and the NAT gateway, the two network pieces that bill by the hour.

1. Type `VPC` in the console search bar and open it.

**AWS console search**

In [ ]:
VPC

2. In its left menu, under **PrivateLink and Lattice**, click **Endpoints**.
3. Tick every endpoint whose type is **Interface**, which is all of them except `leviathan-vpce-s3`. Choose **Actions → Delete VPC endpoints**, type `delete` and click **Delete**.

**VPC endpoint deletion confirmation**

In [ ]:
delete

![vpce_delete.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/002/original/vpce_delete.png?1791359556)

4. In the same left menu, under **Virtual private cloud**, click **NAT gateways**.
5. Select `leviathan-regional-nat`, choose **Actions → Delete NAT gateway**, type `delete` and click **Delete**.

**NAT gateway name**

In [ ]:
leviathan-regional-nat

**NAT gateway deletion confirmation**

In [ ]:
delete

![nat_delete.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/976/original/nat_delete.png?1791359496)

6. A few minutes later, click **Elastic IPs** in the same group. The address AWS gave the NAT gateway goes away by itself.

What is left now costs nothing or almost nothing.

| Left behind | Costs | To remove it |
|---|---|---|
| The space, stopped | Its 5 GB disk, a few cents a month | Delete the space in Studio |
| The domain and user profile | Nothing | Delete spaces, then the user profile, then the domain |
| The VPC, its subnets, security groups and S3 endpoint | Nothing | After the domain is gone, delete the S3 endpoint, then the VPC |
| The role | Nothing | Delete it in IAM after the domain is gone |
| The bucket | About 1 MB of storage | Empty it, then delete it |
| The training and batch jobs | Nothing | They cannot be deleted |

If you are done with SageMaker in this account, remove those too, in that order. A domain cannot be deleted while it still has user profiles, a user profile cannot be deleted while it still has spaces, and a VPC cannot be deleted while anything is still inside it.

## **7. Class summary**

SageMaker is a set of managed machines that meet in S3. `train.py` asked for a machine, a container and two channels, and got back one `model.tar.gz`. Serving it took three objects, a model, a configuration and an endpoint, and only the endpoint bills. Real-time endpoints answer in milliseconds and bill all day. Serverless endpoints bill only for work but took 12 seconds to wake up for a caller on the public URL. Batch transform needs no endpoint at all when nobody is waiting for the answer. The same model file gave the same probability through all three.

All of it ran from Python files on a private platform, and the whole platform came back from an empty account with one script, and its costly parts went away again at the end. That is the shape of real work. The platform is written down, and the model work is code that can be reviewed and run again.

The extras are what turn a working endpoint into one a team can run. Spot capacity cut the training bill by 61% for the same model. CloudWatch counted every request and error without any setup. A Lambda function behind an API Gateway URL gave the model a public address, turned the silently shifted row into a clear 400, and a throttle capped what a flood could send. A blue/green update swapped the model while calls kept coming, a target tracking policy added a machine when traffic stayed high, and a broken release failed without taking the endpoint down.

The quiet failures were the lessons. A role that could see a file but not read it failed a training job only after billing it. The endpoint shifted every value in a row with a missing column and returned a confident wrong answer. The public URL answered a missing permission with a bare 500, and Lambda's 3-second timeout did the same to a cold endpoint. Only the function's log named either cause. A failed release left the endpoint saying InService, and only the failure reason told the truth. And the SDK itself had a bug that a drop down to boto3 got around. None of these raised a clear error at the moment it mattered, and catching them came down to reading the numbers and the logs.

## **8. References and further reading**

**Official documentation**

- [Train a model with Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-training.html). What a training job does with your channels, your container and your output path.
- [SageMaker Python SDK](https://sagemaker.readthedocs.io/). `ModelTrainer`, the resource classes and every other call used in the Python files.
- [XGBoost algorithm with Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html). Every hyperparameter, and the input format for inference.
- [Managed spot training](https://docs.aws.amazon.com/sagemaker/latest/dg/model-managed-spot-training.html). How the wait limit and checkpoints work when AWS takes a machine back.
- [Deployment guardrails](https://docs.aws.amazon.com/sagemaker/latest/dg/deployment-guardrails.html). Blue/green updates, all at once, canary and linear traffic shifting, and automatic rollback.
- [Automatic scaling of SageMaker AI models](https://docs.aws.amazon.com/sagemaker/latest/dg/endpoint-auto-scaling.html). Target tracking and the other policy types.
- [Monitor SageMaker AI with CloudWatch](https://docs.aws.amazon.com/sagemaker/latest/dg/monitoring-cloudwatch.html). Every metric an endpoint reports and what it means.
- [Real-time inference](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html), [Serverless Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/serverless-endpoints.html) and [Batch transform](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html). One page for each serving option.
- [NAT gateways](https://docs.aws.amazon.com/vpc/latest/userguide/vpc-nat-gateway.html). Regional NAT gateways and the routes that use them.

**Pricing**

- [Amazon SageMaker AI pricing](https://aws.amazon.com/sagemaker/ai/pricing/), [Amazon VPC pricing](https://aws.amazon.com/vpc/pricing/) for NAT gateways, and [AWS PrivateLink pricing](https://aws.amazon.com/privatelink/pricing/) for endpoints. Prices change, so check these before relying on the numbers here.